# wh_poc — 장소 추천 에이전트 (멀티턴)

## 구조

```
Layer 0  공용 유틸        dump / _run_id / _load_json / _log / _track / _timed_tool
Layer 1  외부 클라이언트   _kakao_keyword / _naver_search / scrape_urls / _llm / _llm_async
Layer 2  툴 (에이전트 노출) get_today / collect_places / collect_info / collect_context / read_file
Layer 3  에이전트          LlmAgent + Chat (세션 유지)
```

## 설계 원칙

- **툴 경계 = 에이전트가 판단해야 하는 지점.** 코드가 관측 가능한 조건(결과 개수, `is_end`,
  API 응답 상태)은 함수 안으로 내리고, 의미론적 판단만 에이전트에게 남긴다.
- **툴은 신호를 주고, 판단은 에이전트가 한다.** `too_broad`는 "상한을 넘겼다"는 사실만 알려주고,
  그게 문제인지(되물을지)는 발화를 이해해야 알 수 있으므로 에이전트 몫이다.
- **모든 툴 결과는 파일로 떨구고 경로만 반환.** 원본이 컨텍스트에 올라가지 않는다.
- **한 곳에서만 쓰는 로직은 함수로 빼지 않는다.**

## 멀티턴에서 달라진 것

| 변경 | 이유 |
|---|---|
| `kind` / `message` 필드 | 추천·되묻기·잡담을 한 스키마로 구분. 프론트가 카드를 띄울지 결정 |
| state에서 `temp:` 제거 | `temp:`는 invocation 스코프라 다음 턴에 날아간다. 파일 경로는 세션 내내 필요 |
| `too_broad` 신호 | `exhausted=False`인데 상한을 채웠다 = 더 있는데 잘랐다 = 범위가 넓다 |
| `Chat` 클래스 | 세션 재사용. 이전 턴의 툴 결과를 에이전트가 참조할 수 있다 |
| `STATUS_TEXT` | 진행 상태를 모델이 지어내게 하지 않고 툴 호출을 번역해 보여준다 |

재시도 카운터(`temp:collect_attempts`)만 `temp:`로 남겼다. 이건 발화마다 리셋되는 게 맞다.

## 관전 포인트

되묻기가 **남발되지 않는지**가 핵심이다. "신림역 백반집"처럼 이미 구체적인 요청에
"어디로 좁힐까요?"를 물으면 서비스가 짜증나진다. 마지막 테스트 셀에서 이걸 확인한다.

---
# Layer 0 — 공용 유틸

여러 툴에서 반복 사용되는 것만 모았다.

In [30]:
import os
import re
import json
import html
import time
import asyncio
import requests
import httpx
import functools
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
from collections import defaultdict
load_dotenv()

TOOL_FILE_DIR = Path("./tool_file")
TOOL_FILE_DIR.mkdir(exist_ok=True)

# 노트북에서 파이프라인 내부를 들여다볼 때 True. 에이전트 실행 시엔 False 권장.
VERBOSE = True

# 동시성 제한 — 실측으로 정한 값
NAVER_SEM = asyncio.Semaphore(10)   # 네이버 API. 10 초과 시 429 빈발
SCRAPE_SEM = asyncio.Semaphore(20)  # 블로그 본문 크롤링
LLM_SEM = asyncio.Semaphore(10)     # LLM 병렬 호출

_FAIL_PREFIX = "__SCRAPE_FAIL__"    # 본문이 "[제목]" 형태로 시작하는 정상 글과 구분하기 위한 마커


def _log(*args):
    """VERBOSE일 때만 출력. 툴 내부 진행상황 확인용."""
    if VERBOSE:
        print(*args)

_STATS = defaultdict(lambda: {"n": 0, "sec": 0.0})


def _track(label: str):
    """Layer 1 함수를 감싸 호출 횟수와 누적 시간을 집계한다."""
    def deco(fn):
        if asyncio.iscoroutinefunction(fn):
            @functools.wraps(fn)
            async def wrapper(*a, **kw):
                t0 = time.perf_counter()
                try:
                    return await fn(*a, **kw)
                finally:
                    _STATS[label]["n"] += 1
                    _STATS[label]["sec"] += time.perf_counter() - t0
            return wrapper
        @functools.wraps(fn)
        def wrapper(*a, **kw):
            t0 = time.perf_counter()
            try:
                return fn(*a, **kw)
            finally:
                _STATS[label]["n"] += 1
                _STATS[label]["sec"] += time.perf_counter() - t0
        return wrapper
    return deco


def _timed_tool(fn):
    """툴 총 소요시간 + 그 안에서 쓰인 Layer 1 함수별 집계를 출력한다."""
    @functools.wraps(fn)
    async def wrapper(*a, **kw):
        _STATS.clear()
        t0 = time.perf_counter()
        try:
            return await fn(*a, **kw)
        finally:
            total = time.perf_counter() - t0
            _log(f"[{fn.__name__}] 총 {total:.1f}초")
            for label, s in sorted(_STATS.items(), key=lambda x: -x[1]["sec"]):
                _log(f"   {label:<12} {s['sec']:6.1f}초  ({s['n']}회, "
                     f"평균 {s['sec']/s['n']:.2f}초)")
    return wrapper

def dump(tool_name: str, data, run_id: str) -> str:
    """툴 결과를 파일로 떨군다. 같은 invocation의 산출물은 run_id로 묶인다."""
    path = TOOL_FILE_DIR / f"{run_id}_{tool_name}.json"
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(path)


def _load_json(file_path: str):
    with open(file_path, encoding="utf-8") as f:
        return json.load(f)


def _strip_tags(text: str) -> str:
    """네이버 응답의 <b> 하이라이트 태그와 HTML 엔티티 제거."""
    return html.unescape(re.sub(r"<[^>]+>", "", text or "")).strip()

In [31]:
from google.adk.tools import ToolContext


def _run_id(tool_context: ToolContext) -> str:
    """현재 invocation 식별자. 같은 요청에서 나온 파일들을 한 덩어리로 묶는다."""
    key = "temp:run_id"
    rid = tool_context.state.get(key)
    if rid is None:
        rid = str(int(time.time() * 1000))
        tool_context.state[key] = rid
    return rid

---
# Layer 1 — 외부 클라이언트

각 API를 감싸는 얇은 래퍼. 툴로 노출하지 않는다.

In [32]:
@_track("kakao")
def _kakao_keyword(query: str, page: int = 1, category_code: str = "") -> tuple[list[dict], bool]:
    """카카오 로컬 키워드 검색 1페이지. (결과, is_end) 반환.

    카카오는 상호명·카테고리·태그 문자열 매칭이라 서술형 검색어에 약하다.
    페이지당 15개, 전체 노출 가능 문서는 45개가 상한.
    """
    params = {"query": query, "size": 15, "page": page}
    if category_code:
        params["category_group_code"] = category_code

    resp = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers={"Authorization": f"KakaoAK {os.environ['KAKAO_REST_API_KEY']}"},
        params=params,
        timeout=5,
    )
    resp.raise_for_status()
    body = resp.json()

    places = [
        {
            "id": d["id"],
            "name": d["place_name"],
            "category": d["category_name"],
            "address": d["road_address_name"] or d["address_name"],
            "phone": d.get("phone", ""),
            "url": d["place_url"],
            "source": "kakao_local",
        }
        for d in body["documents"]
    ]
    return places, body["meta"]["is_end"]

In [33]:
@_track("naver")
async def _naver_search(query: str, kind: str = "blog", display: int = 10,
                        sort: str = "sim") -> tuple[list[dict], str]:
    """네이버 검색 API 비동기 호출. blog / news / webkr 모두 같은 응답 구조라 하나로 처리.

    kind: 'blog'(블로그) / 'news'(뉴스) / 'webkr'(웹문서)
    sort: 'sim'(정확도) / 'date'(최신). webkr은 sort 미지원이라 무시된다.
    반환: (결과 리스트, 상태) — 상태는 'ok' / 'empty' / '429'
    """
    async with NAVER_SEM:
        for attempt in range(3):
            async with httpx.AsyncClient(timeout=5) as client:
                resp = await client.get(
                    f"https://openapi.naver.com/v1/search/{kind}",
                    headers={
                        "X-Naver-Client-Id": os.environ["NAVER_CLIENT_ID"],
                        "X-Naver-Client-Secret": os.environ["NAVER_CLIENT_SECRET"],
                    },
                    params={"query": query, "display": display, "sort": sort},
                )
            if resp.status_code == 429:
                await asyncio.sleep(0.5 * (attempt + 1))
                continue
            resp.raise_for_status()
            break
        else:
            return [], "429"

    items = resp.json().get("items", [])
    if not items:
        return [], "empty"

    # 필드명이 종류별로 다르다: 블로그는 postdate, 뉴스는 pubDate, 웹문서는 날짜 없음
    return [
        {
            "title": _strip_tags(i.get("title", "")),
            "description": _strip_tags(i.get("description", "")),
            "link": i.get("link", ""),
            "date": i.get("postdate", "") or i.get("pubDate", ""),
        }
        for i in items
    ], "ok"

In [34]:
_BODY_SELECTORS = [
    "div.se-main-container",   # 네이버 스마트에디터 ONE
    "#postViewArea",           # 네이버 구버전 에디터
    "div#post-view",
    "article",                 # 일반 웹페이지
    "main",
]


async def scrape_urls(urls: list[str], max_chars: int = 0) -> list[dict]:
    """URL 병렬 크롤링 → [{'url', 'text'}]. 실패해도 예외 대신 마커 문자열을 담는다.

    실패 판정은 text.startswith(_FAIL_PREFIX)로 한다.
    max_chars가 0이면 본문 전체를 담는다.
    """
    from bs4 import BeautifulSoup

    ua = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
          "(KHTML, like Gecko) Chrome/120.0 Safari/537.36")
    @_track("scrape")
    async def fetch_one(client, url: str) -> str:
        # 네이버 블로그는 iframe 껍데기라 PostView로 우회해야 본문이 나온다
        m = re.search(r"blog\.naver\.com/([^/?]+)/(\d+)", url)
        target = (f"https://blog.naver.com/PostView.naver?blogId={m.group(1)}&logNo={m.group(2)}"
                  if m else url)

        async with SCRAPE_SEM:
            try:
                resp = await client.get(target)
                resp.raise_for_status()
            except httpx.HTTPError as e:
                return f"{_FAIL_PREFIX}열기 실패: {type(e).__name__}"

        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        # 셀렉터를 순회하되, 잡히긴 했는데 내용이 짧으면(광고 div 등) 다음 것으로 넘어간다
        text = ""
        for selector in _BODY_SELECTORS:
            node = soup.select_one(selector)
            if node:
                candidate = re.sub(r"\s+", " ", node.get_text(separator=" ", strip=True)).strip()
                if len(candidate) > 150:
                    text = candidate
                    break
        if not text:
            text = re.sub(r"\s+", " ", (soup.body or soup).get_text(separator=" ", strip=True)).strip()

        if len(text) < 150:
            return f"{_FAIL_PREFIX}본문 추출 실패"
        return text[:max_chars] if max_chars else text

    async with httpx.AsyncClient(
        headers={"User-Agent": ua}, timeout=10.0, follow_redirects=True
    ) as client:
        texts = await asyncio.gather(*(fetch_one(client, u) for u in urls))

    return [{"url": u, "text": t} for u, t in zip(urls, texts)]

In [35]:
from google import genai
from google.genai import types

_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
UTIL_MODEL = "gemini-3.1-flash-lite"   # 툴 내부 유틸용. 에이전트 모델과 별개로 교체 가능


def _llm(prompt: str, *, json_mode: bool = False, temperature: float = 0.0) -> str:
    """유틸 LLM 동기 호출. 단발 처리용."""
    config = types.GenerateContentConfig(temperature=temperature)
    if json_mode:
        config.response_mime_type = "application/json"
    resp = _client.models.generate_content(model=UTIL_MODEL, contents=prompt, config=config)
    return resp.text

@_track("llm")
async def _llm_async(prompt: str, *, json_mode: bool = False, temperature: float = 0.0) -> str:
    """유틸 LLM 비동기 호출. 병렬 처리용."""
    config = types.GenerateContentConfig(temperature=temperature)
    if json_mode:
        config.response_mime_type = "application/json"
    resp = await _client.aio.models.generate_content(
        model=UTIL_MODEL, contents=prompt, config=config
    )
    return resp.text

---
# Layer 2 — 툴

에이전트에 노출되는 함수들. docstring이 곧 LLM에게 주는 명세다.

In [36]:
def get_today() -> dict:
    """오늘 날짜와 요일, 현재 시각을 알려준다.

    '오늘', '이번 주말', '지금 하는' 같은 상대적 시점이 요청에 포함되거나,
    축제·행사처럼 기간이 있는 대상을 찾을 때 먼저 호출해 현재 시점을 확인한다.

    Returns:
        {"date": "2026-08-13", "weekday": "목요일", "time": "14:30", "year": 2026, "month": 8}
    """
    now = datetime.now(ZoneInfo("Asia/Seoul"))
    weekdays = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"]
    return {
        "date": now.strftime("%Y-%m-%d"),
        "weekday": weekdays[now.weekday()],
        "time": now.strftime("%H:%M"),
        "year": now.year,
        "month": now.month,
    }

In [37]:
_EXTRACT_NAMES_PROMPT = """\
아래는 네이버 블로그 검색 결과다. 각 줄은 '제목 | 요약' 형식이다.
여기서 실제 존재하는 가게/장소의 고유 상호명만 뽑아라.
- 제목에 상호명이 있을 확률이 높다. 요약은 보조로만 참고한다.
- '카페 추천', '맛집 베스트10', '데이트코스' 같은 일반명사구는 상호명이 아니다.
- 같은 매장이 여러 줄에 걸쳐 나오면 한 번만 적어라.
- 확실하지 않으면 넣지 마라.
- 결과는 문자열 배열 JSON으로만 출력한다. 예: ["블루보틀 성수", "어니언 성수"]
"""

MIN_COUNT = 20      # 이 미만이면 부족하다고 보고 재시도/폴백
MAX_COUNT = 40      # 이만큼 모이면 수집 종료
MAX_ATTEMPTS = 3

@_timed_tool
async def collect_places(
    query: str,
    tool_context: ToolContext,
    category_code: str = "",
) -> dict:
    """지역 키워드로 장소 후보 목록을 수집한다. 모든 작업의 첫 단계다.

    카카오 로컬 검색 기반이라 이름·주소·좌표는 정확하지만, 영업시간이나 행사 일정처럼
    변하는 정보는 담기지 않는다. 그런 정보가 필요하면 이 툴 다음에 collect_context를 쓴다.

    검색어는 '지역 + 짧은 명사' 형태여야 한다. 서술형 문장은 결과가 거의 나오지 않는다.
      좋은 예: '성수동 카페', '진해 군항제', '가평 캠핑장'
      나쁜 예: '성수동에서 조용히 작업하기 좋은 카페'

    Args:
        query: 검색어. '지역 + 짧은 명사' 조합.
        category_code: 카테고리 필터. 'CE7'=카페, 'FD6'=음식점, 'AT4'=관광명소, 'CT1'=문화시설.
                       확실하지 않으면 빈 문자열로 두는 편이 결과가 많다.

    Returns:
        status가 'retry'면 결과가 부족한 것이므로, message 안내대로 query를 바꿔 다시 호출한다.
        status가 'ok'면 수집이 끝난 것이므로 다음 단계로 넘어간다.
        status가 'too_broad'면 후보가 상한을 넘은 것이다. message를 읽고 그대로 진행할지
        사용자에게 좁힐 기준을 되물을지 판단한다.
        축제나 특정 시설처럼 대상이 하나뿐인 경우 결과가 몇 개여도 'ok'가 나온다. 정상이다.
        예시(완료): {'status': 'ok', 'count': 40, 'file': './tool_file/..._collect_places.json'}
    """
    attempt_key = "temp:collect_attempts"
    attempts = tool_context.state.get(attempt_key, 0) + 1
    tool_context.state[attempt_key] = attempts

    places: dict[str, dict] = {}

    # 1) 카카오 로컬 페이지네이션 (id 기준 dedup)
    #    is_end로 "카카오가 가진 걸 다 줬는지"를 구분한다.
    #    결과가 적은 것과 더 이상 없는 것은 완전히 다른 상태다.
    page = 1
    exhausted = False
    while len(places) < MAX_COUNT and page <= 45:
        batch, is_end = _kakao_keyword(query, page=page, category_code=category_code)
        for p in batch:
            places.setdefault(p["id"], p)
        if is_end:
            exhausted = True
            break
        page += 1
    _log(f"[collect_places] '{query}' 카카오 {len(places)}개 "
         f"(시도 {attempts}/{MAX_ATTEMPTS}, exhausted={exhausted})")

    # 2) 부족한데 더 나올 여지가 있으면 → 에이전트에게 쿼리 재구성 요청
    #    같은 쿼리 재시도는 같은 결과이므로, 어휘를 바꾸는 판단은 LLM 몫이다.
    #    exhausted면 무엇을 해도 결과가 같으므로 재시도하지 않는다.
    if len(places) < MIN_COUNT and attempts < MAX_ATTEMPTS and not exhausted:
        return {
            "status": "retry",
            "message": (
                f"'{query}' 검색 결과가 {len(places)}개로 부족합니다(최소 {MIN_COUNT}개 필요). "
                f"({attempts}/{MAX_ATTEMPTS}번째 시도) "
                "지역명은 유지하고 업종어를 카카오에 등록될 법한 다른 표현으로 바꿔 다시 호출하세요. "
                "예: '백반집' → '밥집' 또는 '한식', '작업 카페' → '카공카페' 또는 '스터디카페'"
            ),
        }

    # 3) 재시도를 다 쓰고도 부족하면 → 블로그에서 상호명을 뽑아 카카오로 실존 검증 후 보충
    #    exhausted면 억지로 채우지 않는다. 축제 검색에 딸려온 주변 식당이 섞이기 때문.
    if len(places) < MIN_COUNT and not exhausted:
        blog_items, _ = await _naver_search(query, kind="blog", display=30, sort="sim")
        lines = "\n".join(f"{i+1}. {b['title']} | {b['description']}"
                           for i, b in enumerate(blog_items))
        try:
            names = json.loads(_llm(f"{_EXTRACT_NAMES_PROMPT}\n{lines}", json_mode=True))
            names = [n.strip() for n in names if isinstance(n, str) and n.strip()]
        except (json.JSONDecodeError, TypeError):
            names = []
        _log(f"[collect_places] 블로그 폴백 → 후보명 {len(names)}개")

        for name in names:
            if len(places) >= MAX_COUNT:
                break
            verified, _ = _kakao_keyword(name, page=1)
            for p in verified:                       # 실존하는 것만, 이름당 1건
                if p["id"] not in places:
                    p["source"] = "blog_derived"
                    places[p["id"]] = p
                    break

    result = list(places.values())[:MAX_COUNT]
    file_path = dump("collect_places", result, _run_id(tool_context))
    tool_context.state["last_places_file"] = file_path
    tool_context.state["last_query"] = query

    payload = {
        "count": len(result),
        "source_breakdown": {
            "local": sum(1 for p in result if p["source"] == "kakao_local"),
            "blog_derived": sum(1 for p in result if p["source"] == "blog_derived"),
        },
        "file": file_path,
    }

    # exhausted가 아닌데 상한을 채웠다 = 더 있는데 잘랐다 = 검색 범위가 넓다.
    # 이 사실만 알려주고, 좁힐지 진행할지 판단은 에이전트에게 맡긴다.
    # (사용자 발화에 이미 기준점이 있는지는 코드가 알 수 없다)
    if not exhausted and len(result) >= MAX_COUNT:
        payload["status"] = "too_broad"
        payload["message"] = (
            f"'{query}' 후보가 {MAX_COUNT}개를 넘어 일부만 가져왔습니다. "
            "사용자 요청에 이미 좁히는 기준(역 이름, 구체적 동네 등)이 있으면 그대로 진행하세요. "
            "기준이 없어 범위가 막연하다면 사용자에게 좁힐 기준을 되물어도 됩니다."
        )
    else:
        payload["status"] = "ok"

    return payload

In [38]:
_SUMMARIZE_PROMPT = """\
사용자 요청: "{user_query}"

아래는 '{place_name}'에 대한 블로그 후기 {n}건이다.
여러 후기를 종합해 위 요청 관점에서 이 장소를 평가하고 다음 JSON 형식으로만 답해라.

{{
  "strengths": ["요청 조건에 부합하는 점 2~4개"],
  "weaknesses": ["요청 조건에 아쉬운 점 또는 주의할 점 0~3개"],
  "summary": "전반적인 평가 2~3문장"
}}

규칙:
- 여러 후기에서 반복되는 내용일수록 신뢰도가 높다. 한 후기에만 나오는 개인적 감상은 비중을 낮춰라.
- 후기 간 내용이 상충하면 그 사실을 그대로 적어라. (예: "시간대에 따라 웨이팅 편차가 큼")
- 본문에 실제로 언급된 내용만 사용하고 추측하지 마라.
- 홍보성 과장 표현은 걷어내고 실제 경험 위주로 판단해라.
- 후기가 이 장소에 대한 내용이 아니거나 정보가 부실하면
  {{"strengths": [], "weaknesses": [], "summary": "정보 부족"}}로 답해라.
"""

@_timed_tool
async def collect_info(tool_context: ToolContext, user_query: str, region_prefix: str) -> dict:
    """수집된 장소들의 블로그 후기를 모아 장점·단점 중심으로 요약한다.

    실제 방문자 경험(맛, 분위기, 혼잡도, 1인석 유무 등)이 판단 근거로 필요할 때 쓴다.
    카페·음식점처럼 방문 후기가 많은 대상에 적합하다.
    축제·행사처럼 '언제 열리는지'가 중요한 대상은 collect_context를 대신 쓴다.

    Args:
        user_query: 사용자의 원래 요청 문장. 어떤 관점에서 장단점을 판단할지 결정한다.
                    예: '신림역 혼밥하기 좋은 백반집'
        region_prefix: 블로그 검색어에 붙일 지역명. 사람들이 실제로 그 동네를 부르는
                    통칭을 쓴다. 도로명이나 행정동명이 아니다.
                    예: 관악구 관천로 일대 → '신림', 금천구 가산디지털2로 → '가산'

    Returns:
        {"count": int, "no_info": int, "url_file": str, "summary_file": str}
        count는 요약이 만들어진 장소 수, no_info는 후기를 못 찾은 장소 수.
        판단 근거는 summary_file에 있다. read_file로 읽어라.
    """
    places_file = tool_context.state.get("last_places_file")
    if not places_file:
        return {"status": "error",
                "message": "collect_places를 먼저 호출하세요. 이 툴을 다시 호출해도 같은 결과입니다."}

    places = _load_json(places_file)
    run_id = _run_id(tool_context)

    # --- 1) 매장별 블로그 URL 수집 ---
    # 도로명은 행정 표기라 실제 리뷰에서 쓰는 생활권 이름과 다르다("관천" vs "신림").
    # 그 매핑은 지역 상식이라 코드가 못 하므로 에이전트가 준 region_prefix를 쓴다.
    # sim/date는 매장마다 유리한 쪽이 달라 사전 판단이 불가능하므로 둘 다 던지고 합친다.
    queries = [f"{region_prefix} {p['name']}".strip() for p in places]

    raw = await asyncio.gather(*[
        _naver_search(q, kind="blog", display=10, sort=s)
        for q in queries for s in ("sim", "date")
    ])

    urls, reasons = {}, {}
    for i, p in enumerate(places):
        merged = raw[i * 2][0] + raw[i * 2 + 1][0]
        statuses = (raw[i * 2][1], raw[i * 2 + 1][1])

        seen, hits = set(), []
        for b in merged:
            if b["link"] in seen:
                continue
            seen.add(b["link"])
            # 검색어가 통해도 무관한 글이 섞이므로 상호명 실제 등장 여부로 한 번 더 거른다
            if p["name"] in b["title"] or p["name"] in b["description"]:
                hits.append(b)

        urls[p["name"]] = hits[:5]
        if not hits:
            reasons[p["name"]] = ("empty" if all(s == "empty" for s in statuses)
                                  else "no_match_after_filter")

    url_file = dump("info_urls", urls, run_id)
    if reasons:
        from collections import Counter
        _log(f"[collect_info] 후기 미확보 {len(reasons)}개 → {dict(Counter(reasons.values()))}")

    # --- 2) 본문 크롤링 (파일로 남기지 않음: 요약이 나오면 원문은 용도 끝) ---
    links, owner = [], []
    for name, items in urls.items():
        for idx, item in enumerate(items):
            links.append(item["link"])
            owner.append((name, idx))

    scraped = await scrape_urls(links)
    bodies: dict[str, list[str]] = {name: [] for name in urls}
    for (name, _), s in zip(owner, scraped):
        if not s["text"].startswith(_FAIL_PREFIX):
            bodies[name].append(s["text"])
    _log(f"[collect_info] 크롤링 {len(scraped)}건 중 성공 {sum(len(v) for v in bodies.values())}건")

    # --- 3) 매장당 후기 전체를 한 번에 넣어 통합 요약 ---
    # 후기를 개별 요약하면 반복 빈도(=신뢰도) 정보가 사라진다.
    async def summarize_one(name: str, texts: list[str]) -> dict:
        joined = "\n\n---\n\n".join(f"[후기 {i+1}]\n{t[:2000]}" for i, t in enumerate(texts))
        prompt = _SUMMARIZE_PROMPT.format(
            user_query=user_query, place_name=name, n=len(texts)) + "\n" + joined
        async with LLM_SEM:
            raw_out = await _llm_async(prompt, json_mode=True)
        try:
            return json.loads(raw_out)
        except json.JSONDecodeError:
            return {"strengths": [], "weaknesses": [], "summary": "정보 부족"}

    names = [n for n, t in bodies.items() if t]
    results = await asyncio.gather(*[summarize_one(n, bodies[n]) for n in names])

    # 주소·URL을 요약에 병합 — 이게 없으면 최종 답변에서 주소를 지어낸다
    meta = {p["name"]: p for p in places}
    summaries = {
        n: {
            **r,
            "review_count": len(bodies[n]),
            "address": meta[n].get("address", ""),
            "category": meta[n].get("category", ""),
            "url": meta[n].get("url", ""),
        }
        for n, r in zip(names, results)
    }
    summary_file = dump("info_summary", summaries, run_id)

    return {
        "count": len(summaries),
        "no_info": len(places) - len(summaries),
        "url_file": url_file,
        "summary_file": summary_file,
    }

In [39]:
_CONTEXT_PROMPT = """\
사용자 요청: "{user_query}"
오늘 날짜: {today}

아래는 '{name}'에 대한 웹문서·뉴스 {n}건이다.
사용자 요청에 답하는 데 필요한 사실 정보만 추려 다음 JSON 형식으로만 답해라.

{{
  "period": "행사 기간이나 운영 시기. 본문에 없으면 빈 문자열",
  "facts": ["확인된 사실 2~4개 (내용, 규모, 요금, 예약 방법 등)"],
  "cautions": ["방문 전 알아야 할 점 0~3개"],
  "summary": "전반적인 설명 2~3문장",
  "freshness": "본문 정보가 올해 기준이면 'current', 지난 연도 기준이면 'outdated', 판단 불가면 'unknown'"
}}

규칙:
- 본문에 실제로 적힌 내용만 사용하고 추측하지 마라.
- 날짜가 나오면 연도를 반드시 확인해라. 오늘 날짜 기준으로 지난 정보면 freshness를 'outdated'로 표기해라.
- 광고성 문구와 목록형 나열 글은 신뢰도를 낮게 봐라.
- 관련 정보가 없으면 {{"period": "", "facts": [], "cautions": [], "summary": "정보 부족", "freshness": "unknown"}}로 답해라.
"""

@_timed_tool
async def collect_context(tool_context: ToolContext, user_query: str, top_n: int = 10) -> dict:
    """수집된 장소들의 일정·운영정보를 웹문서와 뉴스에서 찾아 정리한다.

    축제·행사·전시처럼 '언제 열리는지'가 핵심이거나, 관광지·캠핑장처럼 예약·요금·시설
    정보가 필요할 때 쓴다. 뉴스는 날짜와 장소가 사실로 명시되어 블로그보다 신뢰도가 높다.

    카페·음식점처럼 방문자 경험이 중요한 대상은 collect_info를 대신 쓴다.

    Args:
        user_query: 사용자의 원래 요청 문장.
        top_n: 상위 몇 개 장소를 조사할지. 축제·행사는 후보가 적으므로 기본값으로 충분하다.

    Returns:
        {"count": int, "no_info": int, "source_file": str, "context_file": str}
        정리된 사실 정보는 context_file에 있다. read_file로 읽어라.
        freshness가 'outdated'인 항목은 지난 연도 정보이므로 주의해서 다뤄라.
    """
    places_file = tool_context.state.get("last_places_file")
    if not places_file:
        return {"status": "error",
                "message": "collect_places를 먼저 호출하세요. 이 툴을 다시 호출해도 같은 결과입니다."}

    places = _load_json(places_file)[:top_n]
    run_id = _run_id(tool_context)
    today = get_today()

    # --- 1) 웹문서 + 뉴스 동시 검색 ---
    # 웹문서는 공식 페이지·관공서 자료에, 뉴스는 일정·규모 사실에 강하다. 둘을 합쳐야 커버가 된다.
    tasks = []
    for p in places:
        region = p.get("address", "").split()[0] if p.get("address") else ""
        tasks.append(_naver_search(f"{region} {p['name']}".strip(), kind="webkr", display=5))
        tasks.append(_naver_search(f"{p['name']} {today['year']}", kind="news", display=5, sort="date"))
    raw = await asyncio.gather(*tasks)

    sources = {}
    for i, p in enumerate(places):
        web_items, _ = raw[i * 2]
        news_items, _ = raw[i * 2 + 1]
        merged = ([{**w, "kind": "web"} for w in web_items]
                  + [{**n, "kind": "news"} for n in news_items])
        # 이름이 실제로 언급된 문서만 채택
        sources[p["name"]] = [d for d in merged
                              if p["name"] in d["title"] or p["name"] in d["description"]]
    source_file = dump("context_sources", sources, run_id)
    _log(f"[collect_context] {len(places)}곳 중 문서 확보 "
         f"{sum(1 for v in sources.values() if v)}곳")

    # --- 2) 본문 크롤링 ---
    links, owner = [], []
    for name, items in sources.items():
        for item in items[:4]:
            links.append(item["link"])
            owner.append(name)

    scraped = await scrape_urls(links, max_chars=4000)
    bodies: dict[str, list[str]] = {name: [] for name in sources}
    for name, s in zip(owner, scraped):
        if not s["text"].startswith(_FAIL_PREFIX):
            bodies[name].append(s["text"])

    # 본문 크롤링이 다 실패해도 검색 스니펫만으로 요약을 시도한다
    # (공식 페이지는 봇 차단이 잦은데, 스니펫에 일정이 그대로 담긴 경우가 많다)
    for name, items in sources.items():
        if not bodies[name] and items:
            bodies[name] = [
                "\n".join(f"{d['title']} | {d['description']} ({d['date']})" for d in items)
            ]
    _log(f"[collect_context] 크롤링 {len(scraped)}건 중 성공 "
         f"{sum(1 for s in scraped if not s['text'].startswith(_FAIL_PREFIX))}건")

    # --- 3) 장소별 사실 정리 ---
    async def summarize_one(name: str, texts: list[str]) -> dict:
        joined = "\n\n---\n\n".join(f"[자료 {i+1}]\n{t}" for i, t in enumerate(texts))
        prompt = _CONTEXT_PROMPT.format(
            user_query=user_query, today=today["date"], name=name, n=len(texts)) + "\n" + joined
        async with LLM_SEM:
            raw_out = await _llm_async(prompt, json_mode=True)
        try:
            return json.loads(raw_out)
        except json.JSONDecodeError:
            return {"period": "", "facts": [], "cautions": [],
                    "summary": "정보 부족", "freshness": "unknown"}

    names = [n for n, t in bodies.items() if t]
    results = await asyncio.gather(*[summarize_one(n, bodies[n]) for n in names])

    meta = {p["name"]: p for p in places}
    contexts = {
        n: {
            **r,
            "source_count": len(bodies[n]),
            "address": meta[n].get("address", ""),
            "category": meta[n].get("category", ""),
            "url": meta[n].get("url", ""),
        }
        for n, r in zip(names, results)
    }
    context_file = dump("context_summary", contexts, run_id)

    return {
        "count": len(contexts),
        "no_info": len(places) - len(contexts),
        "source_file": source_file,
        "context_file": context_file,
    }

In [40]:
def read_file(file_path: str) -> dict:
    """툴이 만든 결과 파일을 읽는다. 최종 답변을 쓰기 전에 반드시 호출한다.

    collect_info의 summary_file, collect_context의 context_file에 판단 근거가 들어있다.
    파일을 읽지 않고 답변을 지어내면 안 된다.

    Args:
        file_path: 이전 툴이 반환한 file / summary_file / context_file 값.

    Returns:
        {"content": ...} 파일 내용.
    """
    path = Path(file_path)
    if not path.exists():
        return {"status": "error", "message": f"파일을 찾을 수 없습니다: {file_path}"}
    if not str(path.resolve()).startswith(str(TOOL_FILE_DIR.resolve())):
        return {"status": "error", "message": "허용되지 않은 경로입니다."}
    try:
        return {"content": _load_json(str(path))}
    except json.JSONDecodeError:
        return {"status": "error", "message": "파일을 읽을 수 없습니다."}

---
# Layer 3 — 에이전트

## 에이전트에게 남긴 판단

코드로 내릴 수 있는 것(페이지네이션, dedup, 폴백 조건, 동시성 제어)은 전부 함수 안으로
내렸다. 에이전트가 실제로 판단하는 건 넷이다.

1. **자연어 → 카카오가 먹는 검색어로 압축** — 의미론적 변환.
2. **재시도 시 어휘 교체 + `region_prefix` 지정** — '백반집'→'밥집', '관천'→'신림'.
3. **`collect_info` / `collect_context` 선택** — 카페는 방문 경험이, 축제는 일정 사실이 근거다.
4. **되물을지 진행할지** — `too_broad` 신호를 받았을 때, 사용자 발화에 이미 좁히는 기준이
   있는지 판단한다. `count=30`은 "신림역 백반집"이든 "서울 카페"든 똑같이 찍히므로
   코드로는 구분할 수 없다.

4번이 이번 버전에서 추가된 것이고, 파이프라인에 **되돌아가는 경로**를 만든다.
되묻고 → 답을 받고 → 1단계부터 다시. 그래프가 고정이 아니게 되는 지점이다.

## 체크포인트를 `collect_places` 뒤에 둔 이유

계측 결과가 결정해줬다.

| 툴 | 소요 | 되돌리기 비용 |
|---|---|---|
| `collect_places` | 0.3초 | 거의 없음 |
| `collect_context` | 8초 | 중간 |
| `collect_info` | 28초 | 큼 |

0.3초짜리 뒤에서 물으면 사용자가 기다린 시간이 없고, 답을 받아 다시 돌려도 0.3초다.
반대로 `collect_info` 뒤에서 물으면 이미 28초를 태운 뒤라 절약되는 게 없다.
그래서 후기가 부실할 때는 되묻지 않고 `message`로 한계만 밝히고 진행한다.

In [12]:
from pydantic import BaseModel, Field
from typing import List


class PlaceCard(BaseModel):
    name: str = Field(description="장소명")
    address: str = Field(description="주소. 파일에 있는 값을 그대로 쓴다")
    category: str = Field(default="", description="카테고리")
    reason: str = Field(description="이 요청에 왜 맞는지 2~3문장")
    highlights: List[str] = Field(description="핵심 강점 2~3개. 짧은 구문으로")
    caution: str = Field(default="", description="주의사항. 없으면 빈 문자열")
    period: str = Field(default="", description="행사 기간. 해당 없으면 빈 문자열")
    url: str = Field(default="", description="카카오맵 상세 페이지 URL")


class FeedOutput(BaseModel):
    """모든 응답이 같은 껍데기를 쓴다. kind로 종류를 구분하고,
    프론트엔드는 kind를 보고 카드를 띄울지 말풍선만 띄울지 결정한다."""

    kind: str = Field(
        description="응답 종류. "
                    "'recommendation'=추천 결과가 있음(places를 채운다) / "
                    "'question'=정보가 부족해 사용자에게 되물음 / "
                    "'chat'=인사·안내·일반 대화, 또는 이미 보여준 결과에 대한 후속 답변"
    )
    message: str = Field(
        description="사용자에게 말풍선으로 보여줄 문장. "
                    "recommendation이면 결과 요약 한두 줄, question이면 물어볼 내용, "
                    "chat이면 답변 전문."
    )
    places: List[PlaceCard] = Field(
        default=[],
        description="추천 장소 5~8곳. kind가 'recommendation'일 때만 채우고 그 외에는 빈 배열."
    )

In [13]:
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner

APP_NAME = "wh_poc"
USER_ID = "user1"
AGENT_MODEL = "gemini-3.1-flash-lite"

INSTRUCTION = """당신은 지역 장소를 찾아 추천하는 어시스턴트입니다.
사용자와 여러 번 주고받으며 원하는 것을 좁혀나갑니다.

# 응답 형식
모든 응답은 kind로 종류를 구분합니다. message는 항상 채웁니다.

- **chat** — 인사, 자기소개, 장소와 무관한 질문, 그리고 이미 보여준 추천에 대한 후속 질문.
  툴을 호출하지 말고 message에만 답하세요. places는 비웁니다.
- **question** — 정보가 부족해 사용자에게 되물을 때. places는 비웁니다.
- **recommendation** — 새로 찾은 추천 결과가 있을 때. message에 요약을 쓰고 places를 채웁니다.

# 되물어야 할 때 / 되묻지 말아야 할 때

되묻습니다(kind='question'):
- 지역이 특정되지 않은 경우. "카페 추천해줘"처럼 어디인지 없으면 검색 자체가 무의미합니다.
  → "어느 지역에서 찾아드릴까요?"
- collect_places가 'too_broad'를 반환했고, 사용자 요청에 좁히는 기준이 없는 경우.
  → "신림동 백반집이 30곳 넘게 있어요. 신림역 주변으로 좁혀드릴까요?"

되묻지 않습니다:
- collect_places가 'retry'를 반환한 경우. 이건 검색어 어휘 문제이니 스스로 바꿔 재시도합니다.
- 'too_broad'여도 요청에 이미 기준점이 있는 경우. 그대로 진행하세요.
    "신림역 백반집" → 역이 기준점 → 진행
    "성수동 카공카페" → 동네와 용도가 명확 → 진행
    "신림동 백반" → 기준점 없음 → 되묻기
    "서울 카페" → 너무 넓음 → 되묻기
- 한 대화에서 이미 두 번 되물었다면 더 묻지 말고 가진 결과로 진행하세요.
- 사용자가 "아무거나", "그냥 보여줘"라고 하면 즉시 진행하세요.

# 작업 순서

## 1단계 — 후보 수집
'지역 + 짧은 명사' 형태의 검색어를 만들어 collect_places를 호출합니다.
  예: "성수동에서 조용히 작업할 카페" → "성수동 카공카페"
  예: "이번 봄에 진해 벚꽃 축제 언제야" → "진해 군항제"

status별 행동:
- 'retry' → message 안내대로 업종어를 다른 표현으로 바꿔 즉시 재호출. 되묻지 않습니다.
- 'too_broad' → 위 기준으로 진행할지 되물을지 판단합니다.
- 'ok' → 다음 단계로. 축제처럼 대상이 하나뿐이면 결과가 적어도 정상입니다.

## 2단계 — 어떤 정보가 필요한지 판단
요청 성격에 따라 하나 또는 둘 다 선택합니다. 이 판단이 답변 품질을 좌우합니다.

- **collect_info** (블로그 후기 요약)
  방문자의 실제 경험이 근거인 경우. 맛, 분위기, 혼잡도, 좌석, 서비스 등.
  카페·음식점·술집 등 상설 매장에 적합합니다.

  region_prefix에는 그 동네를 사람들이 실제로 부르는 짧은 이름만 넣으세요.
  '역', '동' 같은 접미사와 도로명·행정동은 빼야 블로그 검색이 맞습니다.
    "신림역" → '신림'   (X: '신림역')
    "관악구 관천로" → '신림'
    "금천구 가산디지털2로" → '가산'

- **collect_context** (웹문서·뉴스 사실 정리)
  일정·기간·요금·예약처럼 확인된 사실이 필요한 경우.
  축제·행사·전시처럼 시기가 중요하거나, 관광지·캠핑장처럼 운영 정보가 필요할 때 씁니다.
  시점이 걸린 요청이면 먼저 get_today로 오늘 날짜를 확인하세요.

애매하면 둘 다 호출해도 됩니다. "캠핑장 추천"은 운영 정보와 이용 후기가 모두 도움이 됩니다.

## 3단계 — 근거 확인
read_file로 summary_file 또는 context_file을 읽습니다.
파일을 읽지 않고 답변을 만들면 안 됩니다.

## 4단계 — 추천 (kind='recommendation')
요청에 가장 잘 맞는 5~8곳을 골라 places에 담습니다.
- review_count 또는 source_count가 많을수록 신뢰도가 높습니다.
- freshness가 'outdated'면 지난 연도 정보이므로 caution에 그 사실을 적으세요.
- address와 url은 파일에 있는 값을 그대로 씁니다. 절대 지어내지 마세요.
- 파일에 없는 정보는 쓰지 마세요. 강점이 부족하면 그 장소를 빼는 편이 낫습니다.
- 후기를 확보한 곳이 적었다면 message에 그 사실을 밝히세요.

# 이어지는 대화

이미 추천을 보여준 뒤 사용자가 후속 질문을 하면, 대화에 남아있는 정보로 답할 수 있는지
먼저 확인하세요.
- "그중에 주차되는 곳은?" → 이미 읽은 파일 내용으로 답변 가능 → kind='chat', 툴 호출 없이 답변
- "다른 곳도 보여줘" / "강남으로 바꿔줘" → 새로 검색 필요 → 1단계부터 다시

같은 정보를 다시 수집하지 마세요. 수집은 시간이 오래 걸립니다.

# 실패 처리
collect_info나 collect_context가 error를 반환하면 collect_places부터 다시 실행하세요.
수집이 실패했는데 collect_places 결과만으로 평가를 지어내면 안 됩니다.
"""

agent = LlmAgent(
    model=AGENT_MODEL,
    name="place_finder",
    description="지역 기반 장소 추천 에이전트",
    instruction=INSTRUCTION,
    tools=[get_today, collect_places, collect_info, collect_context, read_file],
    output_schema=FeedOutput,
)

runner = InMemoryRunner(agent=agent, app_name=APP_NAME)

In [14]:
import uuid

# 툴 호출을 사용자에게 보여줄 상태 문구로 번역한다.
# 모델이 매번 멘트를 지어내게 하는 것보다 일관되고, 토큰도 쓰지 않는다.
STATUS_TEXT = {
    "get_today": "날짜를 확인하고 있어요",
    "collect_places": "장소를 찾고 있어요",
    "collect_info": "방문 후기를 모으는 중이에요",
    "collect_context": "일정과 운영 정보를 확인하고 있어요",
    "read_file": "내용을 정리하고 있어요",
}


class Chat:
    """세션을 유지하며 여러 턴을 주고받는다.

    ADK 세션이 대화 이력을 들고 있으므로, 에이전트는 이전 턴의 툴 결과를 참조할 수 있다.
    같은 Chat 인스턴스를 계속 쓰면 이어지는 대화가 되고, 새로 만들면 처음부터다.
    """

    def __init__(self, show_tools: bool = True, show_status: bool = False):
        self.session_id = str(uuid.uuid4())
        self.show_tools = show_tools
        self.show_status = show_status
        self.history: list[dict] = []
        self._ready = False

    async def _ensure_session(self):
        if not self._ready:
            await runner.session_service.create_session(
                app_name=APP_NAME, user_id=USER_ID, session_id=self.session_id
            )
            self._ready = True

    async def send(self, text: str) -> dict:
        await self._ensure_session()
        content = types.Content(role="user", parts=[types.Part(text=text)])

        out = {"query": text, "turn": len(self.history) + 1,
               "tool_calls": [], "files": {}, "final_text": "", "result": None}

        async for event in runner.run_async(
            user_id=USER_ID, session_id=self.session_id, new_message=content
        ):
            for call in event.get_function_calls():
                out["tool_calls"].append({"name": call.name, "args": dict(call.args)})
                if self.show_status and call.name in STATUS_TEXT:
                    print(f"… {STATUS_TEXT[call.name]}")
                if self.show_tools:
                    print(f"[CALL] {call.name}({dict(call.args)})")

            for fr in event.get_function_responses():
                r = fr.response
                if isinstance(r, dict):
                    for k in ("file", "url_file", "summary_file",
                              "source_file", "context_file"):
                        if k in r:
                            out["files"][k] = r[k]
                    if self.show_tools:
                        brief = {k: v for k, v in r.items() if k != "content"}
                        if "content" in r:
                            brief["content"] = f"<{len(str(r['content']))}자>"
                        print(f"[RESP] {fr.name} → {brief}")

            if event.is_final_response() and event.content and event.content.parts:
                out["final_text"] = "".join(
                    p.text for p in event.content.parts if p.text).strip()

        try:
            out["result"] = json.loads(out["final_text"])
        except (json.JSONDecodeError, TypeError):
            out["result"] = None

        self.history.append(out)
        return out

    async def say(self, text: str) -> dict:
        """send + 사람이 읽기 좋은 출력. 노트북에서 대화를 이어갈 때 쓴다."""
        print(f"\n{'='*60}\n👤 {text}\n{'='*60}")
        out = await self.send(text)
        r = out["result"]
        if not r:
            print(f"\n🤖 (파싱 실패)\n{out['final_text'][:300]}")
            return out

        kind = r.get("kind", "?")
        print(f"\n🤖 [{kind}] {r.get('message', '')}")
        for p in r.get("places", []):
            print(f"\n  ■ {p['name']}  ({p['address']})")
            print(f"    {p['reason']}")
            print(f"    {p['highlights']}")
            if p.get("period"):
                print(f"    기간: {p['period']}")
            if p.get("caution"):
                print(f"    주의: {p['caution']}")
        return out

---
# 테스트

각 계층을 독립적으로 확인한다. 위에서부터 순서대로 실행하면 앞 단계 결과를 뒷 단계가 쓴다.

## Layer 1 — 클라이언트 단위 확인

In [15]:
# 카카오: 상설 매장 vs 축제. 축제는 결과가 적게 나오는 게 정상이다.
for q in ["신림역 밥집", "진해 군항제"]:
    places, is_end = _kakao_keyword(q)
    print(f"{q}: {len(places)}개 (is_end={is_end})")
    for p in places[:3]:
        print(f"   {p['name']} | {p['category']} | {p['address']}")

신림역 밥집: 15개 (is_end=False)
   신림춘천집 본점 | 음식점 > 한식 > 육류,고기 > 닭요리 | 서울 관악구 남부순환로 1600-8
   원조민속순대타운 | 음식점 > 한식 > 순대 | 서울 관악구 신림로59길 14
   막불감동 | 음식점 > 한식 > 국수 | 서울 관악구 남부순환로 1599
진해 군항제: 6개 (is_end=True)
   진해군항제 | 이슈 > 이벤트 > 페스티벌 | 경남 창원시 진해구 평안동 10
   여좌천 | 여행 > 관광,명소 > 하천 | 경남 창원시 진해구 여좌동
   이충무공선양군항제위원회 | 사회,공공기관 > 단체,협회 | 경남 창원시 진해구 백구로 41-1


In [16]:
# 네이버 3종 비교. 같은 쿼리에 소스별로 뭐가 나오는지 눈으로 본다.
for kind in ["blog", "news", "webkr"]:
    items, status = await _naver_search("진해 군항제", kind=kind, display=3, sort="date")
    print(f"\n--- {kind} ({status}) ---")
    for i in items:
        print(f"  {i['title'][:60]}  [{i['date'][:10]}]")


--- blog (ok) ---
  진해군항제, 한국 대표 축제 평가 `종합 2위` 쾌거:창원일보  [20260813]
  진해군항제, 한국 대표 축제 평가 종합 2위 쾌거  [20260812]
  진해 자영업자대출 당일일수 정보 먼저 쉽게 받아보기  [20260812]

--- news (ok) ---
  해사 통합·이전 추진에 진해 주민들 “전면 투쟁”  [Thu, 13 Au]
  진해군항제, 전국 축제 ‘TOP2’  [Thu, 13 Au]
  해군사관학교 통합·이전 추진에 들끓는 창원 진해[현장]  [Thu, 13 Au]

--- webkr (ok) ---
  분야별정보 - 창원특례시청  []
  진해 군항제 - 나무위키  []
  진해군항제 | 문화관광축제 | 대한민국 구석구석 축제  []


In [17]:
# 크롤링: 실패는 예외가 아니라 마커 문자열로 돌아온다.
items, _ = await _naver_search("봉천 도화불백", kind="blog", display=3, sort="date")
scraped = await scrape_urls([i["link"] for i in items])
for s in scraped:
    ok = not s["text"].startswith(_FAIL_PREFIX)
    print(f"[{'OK ' if ok else 'FAIL'}] {len(s['text'])}자 | {s['text'][:80]}")

[OK ] 1076자 | 도화불백 신림본점 서울특별시 관악구 봉천로 304 1층 도화불백 신림본점 오늘은 고기 한 점에 밥 한 숟가락 딱 좋은 곳, 도화불백에 다녀왔습니
[OK ] 595자 | 어제는 #도화불백 #도화불백봉천 에 갔다.. 직원의 추천으로 가본 불백이랑 제육의 맛집이라고한다~! ​ 입장은 2시에 했으며 시간대가 시간대인만
[OK ] 1374자 | ​ 신림 봉천 맛집 도화불백 ​ ⏰ 영업시간 매일 11:30 ~ 21:00 (라스트 오더 20:30) 첫째 주, 셋째 주 토요일 휴무 ​ 🚗 주


In [18]:
# LLM 병렬 동작 확인. 5개가 1개 호출 시간과 비슷하면 정상.
start = time.time()
await asyncio.gather(*[_llm_async("1+1은? 숫자만") for _ in range(5)])
print(f"5개 병렬: {time.time() - start:.1f}초")

5개 병렬: 1.1초


## Layer 2 — 툴 단위 확인

툴은 `tool_context`를 받으므로 노트북에서 직접 부르려면 대역이 필요하다.

In [19]:
class FakeToolContext:
    """ADK가 주입하는 ToolContext 대역. 같은 인스턴스를 재사용하면 state가 유지되어
    재시도 카운트나 파일 경로 전달을 그대로 재현할 수 있다."""
    def __init__(self):
        self.state = {}


ctx = FakeToolContext()
print(get_today())

{'date': '2026-08-13', 'weekday': '목요일', 'time': '21:36', 'year': 2026, 'month': 8}


In [20]:
# collect_places — 결과가 부족하면 retry, 카카오가 다 소진되면(exhausted) 바로 ok.
r = await collect_places("신림역 밥집", ctx)
print(r)

[collect_places] '신림역 밥집' 카카오 45개 (시도 1/3, exhausted=True)
[collect_places] 총 0.3초
   kakao           0.3초  (3회, 평균 0.09초)
{'count': 40, 'source_breakdown': {'local': 40, 'blog_derived': 0}, 'file': 'tool_file\\1786624590038_collect_places.json', 'status': 'ok'}


In [21]:
# retry 흐름 재현: 일부러 안 잡히는 검색어로 시작해 어휘를 바꿔가며 3회까지 가본다.
ctx2 = FakeToolContext()
for q in ["신림역 조용한 작업 백반집", "신림역 백반집", "신림역 밥집"]:
    res = await collect_places(q, ctx2)
    print(res.get("status"), "|", q)

[collect_places] '신림역 조용한 작업 백반집' 카카오 0개 (시도 1/3, exhausted=True)
[collect_places] 총 0.1초
   kakao           0.1초  (1회, 평균 0.08초)
ok | 신림역 조용한 작업 백반집
[collect_places] '신림역 백반집' 카카오 45개 (시도 2/3, exhausted=True)
[collect_places] 총 0.2초
   kakao           0.2초  (3회, 평균 0.07초)
ok | 신림역 백반집
[collect_places] '신림역 밥집' 카카오 45개 (시도 3/3, exhausted=True)
[collect_places] 총 0.3초
   kakao           0.3초  (3회, 평균 0.10초)
ok | 신림역 밥집


In [22]:
# collect_info — 위 collect_places가 남긴 파일을 이어받는다.
# region_prefix는 도로명이 아니라 사람들이 실제로 쓰는 지역 통칭을 넣는다.
info = await collect_info(ctx, "신림역 혼밥하기 좋은 백반집", "신림")
print(info)

summaries = _load_json(info["summary_file"])
for name, s in list(summaries.items())[:3]:
    print(f"\n=== {name} ({s['review_count']}건) | {s['address']}")
    print("  강점:", s["strengths"])
    print("  약점:", s["weaknesses"])
    print("  요약:", s["summary"])

[collect_info] 후기 미확보 4개 → {'no_match_after_filter': 4}
[collect_info] 크롤링 179건 중 성공 179건
[collect_info] 총 29.3초
   scrape       1177.9초  (179회, 평균 6.58초)
   naver         300.6초  (80회, 평균 3.76초)
   llm            69.6초  (36회, 평균 1.93초)
{'count': 36, 'no_info': 4, 'url_file': 'tool_file\\1786624590038_info_urls.json', 'summary_file': 'tool_file\\1786624590038_info_summary.json'}

=== 신림춘천집 본점 (5건) | 서울 관악구 남부순환로 1600-8
  강점: ['신림역에서 도보 1분 거리로 접근성이 매우 뛰어남', '직원이 닭갈비를 직접 볶아주어 혼자서도 편하게 식사 가능', '셀프바를 통해 쌈채소와 밑반찬을 눈치 보지 않고 자유롭게 이용 가능', '태블릿 키오스크를 통한 비대면 주문 방식']
  약점: ['테이블 간격이 좁고 책상이 작아 혼자 식사 시 짐 보관이나 공간 활용이 다소 불편할 수 있음', '피크 타임에는 웨이팅이 발생할 수 있음', '주로 2인 이상 세트 메뉴 위주로 구성되어 있어 1인분 주문 가능 여부를 사전에 확인해야 함']
  요약: 신림역 인근에서 접근성이 좋고 직원이 직접 조리해 주는 편리함 덕분에 혼밥하기에 나쁘지 않은 환경입니다. 다만 매장이 붐비는 시간대에는 테이블이 좁고 대기가 있을 수 있어, 혼잡한 시간을 피해 방문하는 것을 권장합니다.

=== 원조민속순대타운 (5건) | 서울 관악구 신림로59길 14
  강점: ['1인분 주문이 가능하여 혼밥에 용이함', '가격 대비 양이 매우 푸짐함', '기본 서비스로 간과 음료수가 제공됨', '신림역에서 도보 3~5분 거리로 접근성이 좋음']
  약점: ['여러 가게가 모여 있는 개방형 구

In [23]:
# collect_context — 축제 케이스. 카카오로 위치를 잡고 웹·뉴스로 일정을 채운다.
# exhausted 처리 덕분에 결과가 적어도 retry 없이 한 번에 ok가 나온다.
ctx3 = FakeToolContext()
print(await collect_places("진해 군항제", ctx3))

context = await collect_context(ctx3, "진해 군항제 언제 열려?")
print(context)

contexts = _load_json(context["context_file"])
for name, c in list(contexts.items())[:3]:
    print(f"\n=== {name} ({c['source_count']}건, {c['freshness']})")
    print("  기간:", c["period"])
    print("  사실:", c["facts"])
    print("  요약:", c["summary"])

[collect_places] '진해 군항제' 카카오 6개 (시도 1/3, exhausted=True)
[collect_places] 총 0.1초
   kakao           0.1초  (1회, 평균 0.05초)
{'count': 6, 'source_breakdown': {'local': 6, 'blog_derived': 0}, 'file': 'tool_file\\1786624619999_collect_places.json', 'status': 'ok'}
[collect_context] 6곳 중 문서 확보 6곳
[collect_context] 크롤링 18건 중 성공 17건
[collect_context] 총 6.4초
   scrape         14.4초  (18회, 평균 0.80초)
   llm             9.0초  (6회, 평균 1.49초)
   naver           5.1초  (12회, 평균 0.42초)
{'count': 6, 'no_info': 0, 'source_file': 'tool_file\\1786624619999_context_sources.json', 'context_file': 'tool_file\\1786624619999_context_summary.json'}

=== 진해군항제 (4건, current)
  기간: 2026.03.27 ~ 2026.04.05
  사실: ['경상남도 창원시 진해구 일원에서 개최되는 국내 최대 벚꽃 축제임', '주요 프로그램으로 군악의장페스티벌, 이충무공 추모대제, 승전행차 퍼레이드 등이 있음', '평소 출입이 제한되는 해군사관학교와 해군진해기지사령부를 축제 기간에 개방함', '대부분의 행사는 무료이나 체리블라썸뮤직페스티벌 등 일부 프로그램은 유료로 운영됨']
  요약: 진해군항제는 충무공 이순신 장군의 얼을 기리고 벚꽃을 즐기는 대한민국 대표 봄 축제입니다. 36만 그루의 왕벚나무가 만개한 진해 일대에서 다양한 문화예술 공연과 군항도시 특유의 군악의장 행사가 펼쳐집니다.

=== 

In [24]:
# read_file — tool_file/ 밖은 막히는지 확인
import glob
latest = sorted(glob.glob("./tool_file/*_info_summary.json"))[-1]
print(read_file(latest)["content"].keys())
print(read_file("./wh_poc.ipynb"))    # 존재하지만 허용 경로 밖

dict_keys(['신림춘천집 본점', '원조민속순대타운', '막불감동', '오첨지', '서울밥집', '정담은보쌈 신림점', '강강술래 신림점', '서울갈비 신림점', '봄날의 햇살', '문성골족발집', '미나리생삼겹', '서울집 신림직영점', '하나감자탕', '신림목포수산', '대원갈비', '깡통닭갈비', '신림황소곱창', '뇌산마을 신림직영점', '황제갈비', '전라도시네마', '전주식당', '본화로', '강남곱창이야기 신림점', '돼봉삼겹살 신림점', '십원집 신림점', '부뚜막', '맛사랑쭈꾸미 신림점', '장인닭갈비 신림점', '신풍루곱창구이', '신림정', '왕주먹', '독도해물탕 신림점', '흥부보쌈', '도하수산', '서림실비순대', '손칼국수잘하는집'])
{'status': 'error', 'message': '허용되지 않은 경로입니다.'}


## Layer 3 — 에이전트 통합

`Chat` 인스턴스 하나가 한 대화다. 같은 인스턴스로 계속 `say()`를 부르면 이어지는 대화가 되고,
새로 만들면 처음부터 시작한다.

In [25]:
VERBOSE = False   # 툴 내부 로그를 끄고 툴 호출 흐름만 본다

# 단발 — 기준점이 명확한 요청. 되묻지 않고 바로 진행해야 정상.
chat = Chat()
_ = await chat.say("신림역에서 혼밥하기 좋은 백반집 추천해줘")


👤 신림역에서 혼밥하기 좋은 백반집 추천해줘


c:\VSCode\wh_poc\.venv\Lib\site-packages\google\adk\tools\set_model_response_tool.py:134: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


[CALL] collect_places({'query': '신림역 백반', 'category_code': 'FD6'})
[RESP] collect_places → {'count': 40, 'source_breakdown': {'local': 40, 'blog_derived': 0}, 'file': 'tool_file\\1786624628284_collect_places.json', 'status': 'ok'}
[CALL] collect_info({'user_query': '신림역 혼밥하기 좋은 백반집', 'region_prefix': '신림'})
[RESP] collect_info → {'count': 28, 'no_info': 12, 'url_file': 'tool_file\\1786624628284_info_urls.json', 'summary_file': 'tool_file\\1786624628284_info_summary.json'}
[CALL] read_file({'file_path': 'tool_file\\1786624628284_info_summary.json'})
[RESP] read_file → {'content': '<12961자>'}
[CALL] set_model_response({'kind': 'recommendation', 'places': [{'name': '사계절백반', 'url': 'http://place.map.kakao.com/891302253', 'category': '한식', 'address': '서울 관악구 신림로22길 14', 'reason': '혼밥을 위한 환경이 아주 잘 조성되어 있습니다. 특히 개별 트레이에 정갈하게 한상이 나와 눈치 보지 않고 깔끔하게 식사하기 좋습니다.', 'highlights': ['혼밥 전용 좌석 보유', '개별 트레이 제공', '가성비 좋은 한상차림']}, {'name': '송씨제육', 'reason': '혼밥 전용 바 좌석이 마련되어 있고 1인분 주문이 가능해 혼자서 제육볶음을 즐기기에 최

In [26]:
# 행사형 — get_today와 collect_context를 고르는지 확인
chat2 = Chat(show_tools=False, show_status=True)
_ = await chat2.say("이번에 진해 군항제 가려는데 언제 하는지랑 볼거리 알려줘")
print("\n호출된 툴:", [c["name"] for c in chat2.history[-1]["tool_calls"]])


👤 이번에 진해 군항제 가려는데 언제 하는지랑 볼거리 알려줘
… 날짜를 확인하고 있어요
… 장소를 찾고 있어요
… 일정과 운영 정보를 확인하고 있어요
… 내용을 정리하고 있어요

🤖 [recommendation] 진해 군항제에 대해 안내해 드립니다. 2026년 진해 군항제는 3월 27일부터 4월 5일까지 열릴 예정입니다.

충무공 이순신 장군의 업적을 기리는 추모대제와 승전행차를 비롯해, 군악의장페스티벌, 블랙이글스 에어쇼 등 군항제만의 특색 있는 볼거리가 풍성합니다. 특히 축제 기간에는 평소 출입이 제한되는 해군사관학교와 해군진해기지사령부가 일반인에게 개방되니 꼭 방문해 보시길 추천합니다. 

벚꽃 명소로는 여좌천과 경화역, 내수면 생태공원 등이 유명하니 동선을 계획하실 때 참고하세요.

  ■ 진해군항제  (경남 창원시 진해구 평안동 10)
    진해군항제의 핵심 정보와 행사 기간, 주요 프로그램을 제공합니다.
    ['이충무공 추모대제 및 승전행차 재현', '군악의장페스티벌 및 블랙이글스 에어쇼 공연', '평소 개방되지 않는 해군사관학교 내부 탐방']
    기간: 2026.03.27 ~ 2026.04.05
    주의: 축제 기간 중 많은 인파로 인한 교통 체증 및 숙소 부족 문제가 발생할 수 있습니다. 방문 전 교통편과 숙소를 미리 확보하시길 권장합니다.

  ■ 여좌천  (경남 창원시 진해구 여좌동)
    진해군항제 방문 시 필수 코스로 꼽히는 벚꽃 명소로, 행사와 함께 즐기기 가장 좋은 곳입니다.
    ['1.5km에 이르는 벚꽃 터널과 로망스다리', '밤에도 즐길 수 있는 아름다운 경관 조명', '진해를 대표하는 최고의 벚꽃 산책 코스']
    기간: 4월
    주의: 축제 기간 중 사람이 매우 많으니 야간 방문 시 안전에 유의하세요.

호출된 툴: ['get_today', 'collect_places', 'collect_context', 'read_file', 'set_model_response']


In [27]:
# 애매한 쿼리 — collect_info / collect_context 중 뭘 고르는지
chat3 = Chat(show_tools=False, show_status=True)
_ = await chat3.say("가평 캠핑장 추천해줘. 시설 좋고 예약 쉬운 곳으로")
print("\n호출된 툴:", [c["name"] for c in chat3.history[-1]["tool_calls"]])


👤 가평 캠핑장 추천해줘. 시설 좋고 예약 쉬운 곳으로
… 날짜를 확인하고 있어요
… 장소를 찾고 있어요
… 일정과 운영 정보를 확인하고 있어요
… 내용을 정리하고 있어요

🤖 [recommendation] 가평 지역 내 시설이 좋고 예약이 체계적인 캠핑장으로 '산장관광지'를 추천해 드립니다. 

가평군시설관리공단에서 직접 관리하고 있어 시설이 깨끗하고 관리 상태가 좋습니다. 전용 웹사이트를 통해 온라인으로 실시간 예약이 가능하여 예약 과정도 매우 편리합니다. 반려동물 동반이 엄격히 금지되니 이 점 꼭 유의하여 방문하시길 바랍니다. 다른 추천지였던 이화원은 식물원으로 캠핑장이 아닌 점 참고 부탁드립니다.

  ■ 산장관광지  (경기 가평군 상면 덕현산장길 71)
    가평군에서 운영하여 시설이 쾌적하고 관리가 잘 되어 있습니다. 특히 온라인 시스템을 통해 누구나 쉽게 예약할 수 있어 추천해 드립니다.
    ['가평군시설관리공단 직접 운영으로 관리 우수', '다양한 캠핑 사이트 및 콘도형 펜션 보유', '온라인 실시간 예약 시스템으로 편리함']
    기간: 연중 운영
    주의: 전 지역 애완동물 동반 입장 절대 금지, 지정된 장소 외 바비큐 사용 금지, 세면도구 직접 지참 필요

호출된 툴: ['get_today', 'collect_places', 'collect_context', 'read_file', 'set_model_response']


## 멀티턴 시나리오

되묻기와 후속 질문이 실제로 동작하는지 본다.
`chat.say()`를 순서대로 실행하면서 `kind`가 어떻게 바뀌는지 관찰한다.

In [28]:
# 시나리오 1 — 범위가 넓어 되묻는 경우
# "신림동 백반"에는 좁히는 기준이 없다. too_broad를 받고 kind='question'이 나와야 정상.
mt = Chat(show_tools=False, show_status=True)
_ = await mt.say("신림동 백반")


👤 신림동 백반
… 장소를 찾고 있어요
… 방문 후기를 모으는 중이에요
… 내용을 정리하고 있어요

🤖 [recommendation] 신림동에서 든든한 백반 한 끼를 드실 수 있는 곳들을 정리해 드립니다. 혼밥하기 좋은 곳부터 노포 스타일까지 다양하게 선별했습니다.

* **사계절백반**: 대학동 인근에 위치하며, 개별 트레이에 정갈한 한 상이 나와 혼밥하기에 아주 좋습니다. 매일 바뀌는 반찬 구성으로 인기가 높습니다.
* **배바우**: 레트로한 분위기에서 정성스러운 집밥을 즐길 수 있는 곳입니다. 제육볶음 등 메인 메뉴와 반찬 모두 만족도가 높다는 후기가 많습니다.
* **삼봉 신림본점**: 가성비를 중시하신다면 추천합니다. 1인분 주문이 가능하며, 공깃밥과 라면사리 무한리필, 계란프라이 셀프 코너가 있어 매우 든든합니다.
* **송씨제육**: 직화 제육 전문점으로, 불향 가득한 고기를 쌈 채소와 함께 푸짐하게 즐길 수 있습니다. 혼밥 전용 좌석이 잘 갖춰져 있습니다.
* **충남식당**: 할머니의 손맛이 느껴지는 노포를 찾으신다면 좋은 선택입니다. 정겨운 집밥 분위기를 느낄 수 있는 곳입니다.

방문 전 영업시간이나 휴무일은 꼭 확인하시는 것을 권장합니다.

  ■ 사계절백반  (서울 관악구 신림로22길 14)
    대학동 지역의 인기 백반집으로, 혼밥하기 좋은 환경과 매일 바뀌는 정갈한 반찬 구성으로 후기가 좋습니다.
    ['혼밥 최적화 트레이 제공', '매일 바뀌는 정갈한 반찬', '높은 가성비']

  ■ 배바우  (서울 관악구 신림로56길 7)
    방문객들의 만족도가 매우 높은 곳으로, 집밥 같은 정성스러운 반찬과 메인 메뉴를 함께 즐길 수 있습니다.
    ['정성스러운 집밥 스타일 반찬', '친절한 서비스', '조용하고 레트로한 분위기']

  ■ 삼봉 신림본점  (서울 관악구 호암로24길 63)
    가성비와 든든함을 동시에 잡은 곳입니다. 라면사리와 밥 무한리필 서비스가 있어 푸짐한 식사가 가능합니다.
    ['공깃밥/라면사리

In [29]:
# 되묻기에 답하기 → 좁혀서 다시 수집 → kind='recommendation'
_ = await mt.say("신림역 근처로 좁혀줘")


👤 신림역 근처로 좁혀줘
… 장소를 찾고 있어요
… 방문 후기를 모으는 중이에요
… 내용을 정리하고 있어요

🤖 [recommendation] 신림역 근처에서 이용하기 좋은 백반집과 든든한 식당들을 추천해 드립니다. 신림역 접근성과 혼밥 가능 여부를 중심으로 선정했습니다.

* **도화불백**: 직화로 구워 불향 가득한 돼지불백과 제육볶음이 대표 메뉴입니다. 깔끔한 밑반찬과 국물이 함께 나와 든든한 한식을 원하시는 분들께 추천하며, 혼밥하기에도 부담 없습니다.
* **송씨제육**: 신림역에서 가까운 곳으로, 불향 입힌 제육과 찌개류를 1인분씩 주문할 수 있어 혼밥족에게 인기가 매우 높습니다. 밥, 반찬, 신선한 쌈 채소를 셀프바에서 마음껏 가져다 드실 수 있습니다.
* **사계절백반**: 인근에서 이미 유명한 백반 맛집으로, 1인 트레이에 깔끔하게 반찬과 메인 요리가 제공됩니다. 매일 반찬이 바뀌어 자주 방문해도 질리지 않는 것이 큰 강점입니다.
* **전주식당**: 신림역에서 도보로 5분 내외 거리에 위치한 노포 식당입니다. 가성비 좋은 생오리고기와 삼겹살 메뉴가 있으며, 새벽부터 영업하여 이른 아침이나 늦은 저녁 식사로도 방문하기 좋습니다.
* **배바우**: 조용하고 편안한 분위기에서 정성스러운 집밥을 즐기고 싶을 때 추천합니다. 친절한 사장님과 정갈한 밑반찬 구성으로 방문객 만족도가 높은 곳입니다.

이곳들은 신림역 인근에서 식사하기 좋은 장소들이니 취향에 맞춰 선택해 보세요!

  ■ 도화불백  (서울 관악구 봉천로 304)
    신림역 인근에서 불향 가득한 직화 불백과 제육을 즐길 수 있는 곳입니다. 깔끔한 구성으로 혼밥과 직장인 식사로 평이 좋습니다.
    ['직화 조리의 풍부한 불향', '깔끔한 밑반찬과 국물 제공', '혼밥에 부담 없는 분위기']

  ■ 송씨제육  (서울 관악구 봉천로 230)
    신림역 인근에서 가성비와 맛을 모두 잡은 곳입니다. 쌈 채소 무한리필과 1인분 주문이 가능해 혼밥족에게 인기가 많습니다.
    ['직화 

In [30]:
# 후속 질문 — 이미 읽은 파일 내용으로 답할 수 있으므로 툴 호출 없이 kind='chat'
_ = await mt.say("그중에 웨이팅 없는 곳은 어디야?")
print("\n호출된 툴:", [c["name"] for c in mt.history[-1]["tool_calls"]])


👤 그중에 웨이팅 없는 곳은 어디야?

🤖 (파싱 실패)
사용자가 추천해 드린 곳 중 웨이팅 상황을 확인해 보겠습니다.

안타깝게도 **도화불백, 송씨제육, 사계절백반** 등은 인기 맛집으로 식사 시간대에 웨이팅이 발생할 수 있다는 후기가 있습니다. 특히 **송씨제육**이나 **사계절백반**은 매장이 협소해 더 혼잡할 수 있습니다.

상대적으로 웨이팅이 덜한 곳은 다음과 같습니다:

* **배바우**: 조용하고 아담한 분위기라 웨이팅보다는 편안하게 식사할 수 있는 곳입니다. 다만 매장이 지하에 있고 눈에 띄지 않는 곳에 있어, 인기 맛집들보다 상대적으로 덜 붐비는 편입니다.
* **전주식

호출된 툴: []


In [31]:
# 새 검색이 필요한 후속 — 다시 1단계부터, kind='recommendation'
_ = await mt.say("아니다 강남역 쪽으로 다시 찾아줘")
print("\n호출된 툴:", [c["name"] for c in mt.history[-1]["tool_calls"]])


👤 아니다 강남역 쪽으로 다시 찾아줘
… 장소를 찾고 있어요
… 방문 후기를 모으는 중이에요
… 내용을 정리하고 있어요

🤖 [recommendation] 강남역 근처에서 혼밥하기 좋고 가성비가 뛰어난 백반 및 한식당들을 추천해 드립니다.

* **강남백반집**: 강남역 9, 10번 출구에서 매우 가깝고 11,900원에 다양한 메뉴를 즐길 수 있는 한식 뷔페입니다. 매일 메뉴가 바뀌어 직장인들에게 인기가 많으며, 혼밥하기에도 아주 편안한 환경입니다.
* **딸부자네불백 강남역9번출구점**: 불향 가득한 불백 정식을 빠르게 즐길 수 있는 곳입니다. 된장찌개, 계란찜, 쌈 채소가 포함된 알찬 구성으로 혼밥족에게 최적화되어 있습니다.
* **영진식당**: 강남역 인근에서 보기 드문 가성비 백반집입니다. 셀프바를 통해 밑반찬을 원하는 만큼 이용할 수 있고, 혼밥부터 반주까지 즐기기 좋은 편안한 분위기입니다.
* **진미식당**: 매일 바뀌는 가정식 백반을 저렴하게 즐길 수 있는 곳입니다. 밥과 반찬을 직접 리필할 수 있어 든든한 식사가 가능하며 혼밥하기에도 부담 없습니다.
* **청목**: 1만 원 내외의 높은 가성비로 정갈한 집밥 스타일을 맛볼 수 있는 곳입니다. 찌개와 숭늉까지 나오는 가정식 구성으로 직장인들에게 인기가 높습니다.

강남역 인근은 점심시간에 매우 혼잡할 수 있으니 가급적 피크 타임을 피해서 방문하시는 것을 추천드립니다!

  ■ 강남백반집  (서울 서초구 서초대로77길 13)
    강남역 인근에서 매일 바뀌는 식단을 가성비 있게 즐길 수 있는 한식 뷔페로, 혼밥과 직장인 점심 식사로 인기가 높습니다.
    ['매일 바뀌는 다양한 메뉴', '넓고 쾌적한 혼밥 공간', '강남역 도보 3분 거리']

  ■ 딸부자네불백 강남역9번출구점  (서울 서초구 서초대로77길 9)
    접근성이 뛰어나고 혼밥하기에 최적화된 곳입니다. 정식 구성이 매우 알차고 주문 후 음식이 빨리 나와 직장인에게 좋습니다.
    ['불향 나는 불백 정식', '된장찌개/계란

In [32]:
# 시나리오 2 — 지역이 아예 없는 경우. 툴을 부르지 말고 되물어야 한다.
mt2 = Chat(show_tools=False)
_ = await mt2.say("카페 추천해줘")
print("호출된 툴:", [c["name"] for c in mt2.history[-1]["tool_calls"]])   # 비어 있어야 정상

_ = await mt2.say("성수동")


👤 카페 추천해줘

🤖 [question] 어느 지역의 카페를 찾고 계신가요? 지역명을 알려주시면 그곳의 분위기 좋은 카페를 추천해 드릴게요!
호출된 툴: ['set_model_response']

👤 성수동

🤖 [recommendation] 성수동에서 분위기 좋고 인기 있는 카페 8곳을 추천해 드립니다. 각각의 특색이 다르니 취향에 맞는 곳을 골라보세요!

  ■ 5TO7  (서울 성동구 서울숲2길 44-13)
    성수동 골목 감성을 잘 살린 곳으로, 서울숲 산책과 연계한 데이트 장소로 매우 추천하는 카페입니다. 수플레 팬케이크가 유명하지만 조리 시간이 다소 걸릴 수 있어 여유로운 방문을 권장합니다.
    ['주택 개조 아늑한 우드톤 인테리어', '폭신한 수플레 팬케이크 맛집', '서울숲 데이트 코스로 제격']

  ■ 노틀던  (서울 성동구 서울숲2길 11)
    감각적인 인테리어와 함께 완성도 높은 디저트를 즐길 수 있는 카페입니다. 매장이 아담하여 데이트나 대화하기 좋지만, 주말에는 웨이팅이 발생할 수 있습니다.
    ['서울숲 카페거리 뷰의 통창 카페', '고급스러운 비주얼의 디저트', '깔끔한 화이트톤의 차분한 분위기']

  ■ 레이어10  (서울 성동구 상원4길 10)
    번잡한 성수동 메인 거리에서 살짝 벗어나 있어 비교적 한적하고 여유로운 분위기에서 시간을 보낼 수 있습니다. 감각적인 인테리어와 디저트 퀄리티가 높아 만족도가 높습니다.
    ['채광이 좋은 세련된 공간', '성수 번화가 대비 비교적 여유로움', '완성도 높은 디저트류 보유']

  ■ 메리노  (서울 성동구 서울숲길 24)
    뜨개질을 좋아하는 분들에게는 최적의 힐링 장소입니다. 이색적인 컨셉과 차분한 분위기 덕분에 혼자만의 시간을 보내거나 친구와 오붓하게 대화하기 좋습니다.
    ['알록달록 뜨개실 테마의 이색 인테리어', '뜨개질을 즐기며 머물기 좋은 아늑함', '이벤트로 랜덤 뜨개실 증정']

  ■ 바이닐 성수  (서울 성동구 아차산로 15-8)

In [33]:
# 시나리오 3 — 잡담. kind='chat'이고 places는 비어야 한다.
mt3 = Chat(show_tools=False)
_ = await mt3.say("안녕 너 뭐하는 애야?")
r = mt3.history[-1]["result"]
print("kind:", r["kind"], "| places:", len(r["places"]),
      "| 툴:", [c["name"] for c in mt3.history[-1]["tool_calls"]])


👤 안녕 너 뭐하는 애야?

🤖 [chat] 안녕하세요! 저는 지역 기반 장소 추천 에이전트, 'place_finder'입니다.

원하시는 지역의 맛집, 카페, 관광 명소 등을 찾아서 상세한 후기와 운영 정보까지 정리해 드리는 역할을 하고 있어요. 

어떤 곳을 찾고 계신가요? 지역과 함께 무엇을 원하시는지 말씀해 주시면 바로 찾아드릴게요!
kind: chat | places: 0 | 툴: ['set_model_response']


In [34]:
# 되묻기 남발 점검 — 기준점이 명확한 요청들에 question이 나오면 안 된다.
for q in ["신림역 백반집", "성수동 카공카페", "진해 군항제"]:
    c = Chat(show_tools=False)
    out = await c.send(q)
    kind = (out["result"] or {}).get("kind", "?")
    flag = "  ← 되물음(과함)" if kind == "question" else ""
    print(f"{q:20} → {kind}{flag}")

신림역 백반집              → recommendation
성수동 카공카페             → recommendation
진해 군항제               → recommendation


In [35]:
# 전체 대화 이력 훑기
for h in mt.history:
    r = h["result"] or {}
    print(f"[{h['turn']}] {h['query'][:30]:32} → {r.get('kind','?'):15} "
          f"places={len(r.get('places', []))} tools={len(h['tool_calls'])}")

[1] 신림동 백반                           → recommendation  places=5 tools=4
[2] 신림역 근처로 좁혀줘                      → recommendation  places=5 tools=4
[3] 그중에 웨이팅 없는 곳은 어디야?               → ?               places=0 tools=0
[4] 아니다 강남역 쪽으로 다시 찾아줘               → recommendation  places=5 tools=4


---
# 남은 과제

| 항목 | 현황 | 메모 |
|---|---|---|
| 되묻기 판단 정확도 | 실측 필요 | flash-lite가 "기준점이 있는가"를 잘 가리는지가 관건. 남발하면 UX가 나빠진다 |
| 되묻기 횟수 제한 | instruction 의존 | "두 번까지"를 프롬프트로만 걸어둠. state 카운터로 강제하는 편이 확실하다 |
| 재실행 낭비 | 미해결 | 좁혀서 재검색하면 겹치는 매장의 후기를 다시 긁는다. `id` 대조로 스킵 가능 |
| 컨텍스트 누적 | 미해결 | 턴이 쌓이면 이전 `read_file` 결과(10KB)가 계속 남는다. 3~4턴 넘으면 성능 저하 우려 |
| `scrape` 평균 7초 | 조사 필요 | 186건을 SEM=20으로 밀면 건당이 느려진다. 세마포어를 낮추는 쪽이 빠를 수 있다 |
| `naver` 평균 3.5초 | 조사 필요 | 검색 API치고 비정상. 429 재시도가 평균을 끌어올린 것으로 보임 |
| 후기 미확보율 | 개선 중 | `sim`+`date` 병합으로 줄었으나 흔한 상호명은 여전히 취약 |
| `category_code` | 에이전트가 임의 지정 | 잘못 넣으면 결과가 급감. 닫을지 검토 |
| 리랭킹 | 에이전트가 직접 선별 | 후보 30개는 컨텍스트에 들어감. 100개 넘으면 별도 스코어링 필요 |
| 세마포어 전역 | 프로세스 단위 공유 | 서버화 시 요청 간 간섭 검토 |

In [21]:
import json, glob, re
from collections import Counter
from pathlib import Path

TOOL_DIR = Path("./poc/service_v3/tool_file")     # 경로 다르면 여기만 수정

def load(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

# 어떤 run들이 있는지 확인
runs = sorted({f.name.split("_")[0] for f in TOOL_DIR.glob("*_collect_places.json")})
print(f"run {len(runs)}개")
for r in runs[-12:]:
    pool = load(TOOL_DIR / f"{r}_collect_places.json")
    has_summary = (TOOL_DIR / f"{r}_info_summary.json").exists()
    print(f"  {r}  pool={len(pool):2}  summary={'O' if has_summary else 'X'}  "
          f"{', '.join(p['name'] for p in pool[:3])}...")

run 60개
  1787116750448  pool=17  summary=O  하우요커피, 갱고반지하, 더로드101 귀산...
  1787116788784  pool=20  summary=O  막불감동, 서울밥집, 신바야시 쇼쿠도...
  1787116821536  pool=20  summary=O  라온하제, 성산옥프리미엄, 카츠미도리...
  1787116865911  pool=20  summary=O  옛날영도집, 동굴집, 명품갈비토반...
  1787116901784  pool=20  summary=O  맥주를구한피자 마산합성점, 피자덕 창원마산점, 맘스터치 피자앤치킨 창원합성점...
  1787116915023  pool=20  summary=O  동부회센타, 이층횟집, 김해횟집...
  1787116951739  pool=20  summary=O  시골장칼국수, 손칼국수잘하는집, 봉평옹심이메밀칼국수...
  1787116976141  pool=20  summary=O  해방촌부대찌개, 옛맛의정부부대찌개, 다리목...
  1787116997741  pool=20  summary=O  서울갈비 신림점, 신림춘천집 본점, 정담은보쌈 신림점...
  1787117021674  pool=20  summary=O  스타벅스 신림타임스트림점, 블러커피, 작은따옴표 도림천점...
  1787117045642  pool=20  summary=O  어사출또 신림역점, 신림목포수산, 비타민수산...
  1787117065046  pool=20  summary=O  장수촌풍천장어직판장, 어사출또 신림역점, 콩두주백 관악직영점...


In [22]:
def diagnose(run_id: str, targets: list[str] = ()):
    """pool 20개 중 몇 개가 info_summary까지 살아남았나."""
    pool = [p["name"] for p in load(TOOL_DIR / f"{run_id}_collect_places.json")]
    sm_path = TOOL_DIR / f"{run_id}_info_summary.json"
    ul_path = TOOL_DIR / f"{run_id}_info_urls.json"
    if not sm_path.exists():
        print(f"{run_id}: info_summary 없음 (collect_info 미실행)")
        return
    summ, urls = load(sm_path), load(ul_path)

    print(f"\n=== {run_id} ===")
    print(f"pool {len(pool)} → summary {len(summ)}  (탈락 {len(pool) - len(summ)})")

    print("\n[탈락한 매장]  블로그 0건이면 검색·필터 실패, 1건 이상이면 크롤링 실패")
    for i, n in enumerate(pool, 1):
        if n not in summ:
            star = " ★" if any(t in n or n in t for t in targets) else ""
            print(f"  {i:2}위 {n:28} 블로그 {len(urls.get(n, []))}건{star}")

    print("\n[살아남은 매장]")
    for i, n in enumerate(pool, 1):
        if n in summ:
            s = summ[n]
            star = " ★" if any(t in n or n in t for t in targets) else ""
            print(f"  {i:2}위 {n:28} 후기{s['review_count']}건 "
                  f"강점{len(s['strengths'])} | {s['summary'][:30]}...{star}")


# 4회 다 실패한 케이스들로 확인
diagnose("1787117045642", ["목포수산"])          # 신림역 횟집
diagnose("1787116997741", ["서울갈비", "왕주먹"]) # 신림역 고기집


=== 1787117045642 ===
pool 20 → summary 18  (탈락 2)

[탈락한 매장]  블로그 0건이면 검색·필터 실패, 1건 이상이면 크롤링 실패
  19위 숙성회가온다 신림본점                  블로그 0건
  20위 보라매셀프 회 수산시장                 블로그 0건

[살아남은 매장]
   1위 어사출또 신림역점                    후기5건 강점4 | 신림역 인근에서 신선한 회를 합리적인 가격에 즐길 수 ...
   2위 신림목포수산                       후기5건 강점4 | 신림목포수산은 두툼하고 신선한 회를 합리적인 가격에 즐... ★
   3위 비타민수산                        후기5건 강점3 | 신원시장 내 위치한 횟집으로, 특히 대방어와 고등어회 ...
   4위 육회바른연어 신림역점                  후기5건 강점4 | 신림역 인근에서 신선한 육회와 연어를 합리적인 가격에 ...
   5위 비타민수산세꼬시                     후기2건 강점4 | 신림역 인근에서 낮술을 즐기기 좋은 넓고 쾌적한 횟집으...
   6위 캐리비안회적                       후기3건 강점4 | 신림역 인근에서 숙성회를 즐기기 좋은 동네 맛집으로, ...
   7위 길회센타                         후기5건 강점4 | 신림역 인근에서 가성비와 맛을 모두 잡은 인기 횟집으로...
   8위 막해                           후기5건 강점4 | 신림역 인근에서 신선한 숙성회와 다양한 해산물을 가성비...
   9위 해인어시장 신림본점                   후기5건 강점4 | 신림역 인근에서 접근성이 좋고 가성비가 뛰어난 횟집으로...
  10위 와사비                          후기5건 강점3 | 신림역 인근에서 숙성회와 사시미를 즐기기에 적합한 맛집...
  11위 

In [23]:
def check_region(run_id: str, region: str):
    """블로그 글에 지역명이 안 나오면 다른 지역 동명 매장일 수 있다."""
    urls = load(TOOL_DIR / f"{run_id}_info_urls.json")
    print(f"\n=== {run_id} / '{region}' 언급 여부 ===")
    bad_total = tot = 0
    for name, hits in urls.items():
        if not hits:
            continue
        off = [h for h in hits
               if region not in h["title"] and region not in h["description"]]
        tot += len(hits); bad_total += len(off)
        if off:
            print(f"\n  {name}  ({len(off)}/{len(hits)}건 미언급)")
            for h in off[:2]:
                print(f"     {h['title'][:58]}")
    print(f"\n  전체 {bad_total}/{tot}건이 '{region}' 미언급")


check_region("1787117045642", "신림")
check_region("1787116915023", "창원")     # 창원 횟집


=== 1787117045642 / '신림' 언급 여부 ===

  전체 0/66건이 '신림' 미언급

=== 1787116915023 / '창원' 언급 여부 ===

  동부회센타  (1/5건 미언급)
     [경남/진해] 내돈내산 진해맛집 동부회센터 포장후기

  두툼회  (1/5건 미언급)
     진해 용원동 횟집 두툼회(또간집)

  백어횟집  (1/5건 미언급)
     [진해 횟집] 백어횟집

  전체 3/75건이 '창원' 미언급


In [24]:
import json, glob
from pathlib import Path
from collections import defaultdict

files = sorted(glob.glob("./poc/service_v3/*.json"))
print("eval 파일:", [Path(f).name for f in files])

# 정답별 집계: (질문, 정답) → [수집 여부 리스트]
pool_hit = defaultdict(list)
final_hit = defaultdict(list)
pool_rank = defaultdict(list)

for f in files:
    ev = json.loads(Path(f).read_text(encoding="utf-8"))
    for c in ev["cases"]:
        if "error" in c:
            continue
        for d in c["detail"]:
            key = (c["question"], d["answer"])
            pool_hit[key].append(d["in_pool"])
            final_hit[key].append(d["in_final"])
            if d["pool_rank"]:
                pool_rank[key].append(d["pool_rank"])

n_run = len(files)
print(f"\n{'='*80}")
print("[한 번도 수집 안 된 정답]  ← 검색어·카카오 커버리지 문제")
print(f"{'='*80}")
for key, hits in sorted(pool_hit.items()):
    if not any(hits):
        print(f"  {key[1]:24} | {key[0][:34]}  ({len(hits)}회 시도)")

print(f"\n{'='*80}")
print("[가끔 수집됨]  ← 검색어가 흔들려서 놓치는 것")
print(f"{'='*80}")
for key, hits in sorted(pool_hit.items()):
    if any(hits) and not all(hits):
        ranks = pool_rank.get(key, [])
        print(f"  {key[1]:24} | 수집 {sum(hits)}/{len(hits)}회, "
              f"잡힐 때 순위 {ranks} | {key[0][:28]}")

print(f"\n{'='*80}")
print("[항상 수집되는데 최종 탈락]  ← 선별 문제")
print(f"{'='*80}")
for key, hits in sorted(pool_hit.items()):
    if all(hits) and not any(final_hit[key]):
        print(f"  {key[1]:24} | 순위 {pool_rank.get(key, [])} | {key[0][:28]}")

print(f"\n{'='*80}")
print("[가끔 최종에 듦]  ← 노이즈 영역")
print(f"{'='*80}")
for key in sorted(pool_hit):
    fh = final_hit[key]
    if any(fh) and not all(fh):
        print(f"  {key[1]:24} | 최종 {sum(fh)}/{len(fh)}회 | {key[0][:28]}")

eval 파일: ['r1.json', 'r2.json', 'r3.json', 'r4.json']

[한 번도 수집 안 된 정답]  ← 검색어·카카오 커버리지 문제
  뜨라또리아 다젠나                | 마산 합성동 피자 맛집  (4회 시도)
  개성순대                     | 창원 가포 맛집 추천  (4회 시도)
  우동한그릇                    | 창원 가포 맛집 추천  (4회 시도)
  귀산옥 해물칼국수                | 창원 귀산 맛집 추천  (4회 시도)
  거영수산in해녀의집               | 창원 횟집 추천해줘  (4회 시도)

[가끔 수집됨]  ← 검색어가 흔들려서 놓치는 것
  정담은보쌈                    | 수집 1/3회, 잡힐 때 순위 [1] | 신림역 근처 부모님 모시고 가기 좋은 식당 추천좀
  등대집                      | 수집 2/4회, 잡힐 때 순위 [17, 17] | 창원 가포 맛집 추천
  청하횟집                     | 수집 3/4회, 잡힐 때 순위 [16, 16, 16] | 창원 귀산 맛집 추천

[항상 수집되는데 최종 탈락]  ← 선별 문제
  서울갈비 신림점                 | 순위 [1, 1, 1, 1] | 신림역 근처 고기집 추천해줘
  목포수산                     | 순위 [2, 3, 2, 2] | 신림역 주변 횟집 해줘
  더로드101 귀산                | 순위 [3, 6, 6, 3] | 창원 귀산 바다가 보이는 커피 맛집
  블루피쉬                     | 순위 [9, 10, 10, 9] | 창원 귀산 바다가 보이는 커피 맛집
  카페모조                     | 순위 [4, 5, 5, 4] | 창원 귀산 바다가 보이는 커피 맛집
  푸른회전문                    | 순위 [14, 14, 14, 14] | 창원 횟

In [25]:
run = "1787116997741"   # 신림역 고기집
pool = load(TOOL_DIR / f"{run}_collect_places.json")
region = "신림"
for i, p in enumerate(pool, 1):
    mark = "★" if region in p["name"] else " "
    print(f"{mark} {i:2}위 {p['name']:26} {p['category'].split('>')[-1].strip()}")

★  1위 서울갈비 신림점                   갈비
★  2위 신림춘천집 본점                   닭요리
★  3위 정담은보쌈 신림점                  족발,보쌈
★  4위 강강술래 신림점                   갈비
   5위 봄날의 햇살                     닭요리
   6위 문성골족발집                     족발,보쌈
   7위 미나리생삼겹                     삼겹살
★  8위 서울집 신림직영점                  삼겹살
★  9위 신림황소곱창                     곱창,막창
  10위 대원갈비                       갈비
  11위 깡통닭갈비                      닭요리
  12위 황제갈비                       갈비
  13위 본화로                        육류,고기
★ 14위 강남곱창이야기 신림점                곱창,막창
★ 15위 돼봉삼겹살 신림점                  삼겹살
★ 16위 십원집 신림점                    육류,고기
  17위 부뚜막                        육류,고기
  18위 신풍루곱창구이                    곱창,막창
★ 19위 신림정                        육류,고기
  20위 왕주먹                        육류,고기


In [26]:
run = "1787116997741"
summ = load(TOOL_DIR / f"{run}_info_summary.json")

for n in ["서울갈비 신림점", "강강술래 신림점", "왕주먹", "본화로"]:
    s = summ[n]
    print(f"\n■ {n}  (후기 {s['review_count']}건 | {s['category'].split('>')[-1].strip()})")
    print(f"  강점: {s['strengths']}")
    print(f"  약점: {s['weaknesses']}")
    print(f"  요약: {s['summary']}")


■ 서울갈비 신림점  (후기 5건 | 갈비)
  강점: ['특제 소스와 우삼겹의 조합이 뛰어나며 중독성 있는 맛으로 평가받음', '고기가 얇아 빠르게 익으며, 직원들이 직접 구워주는 서비스 제공', '신림역에서 도보로 이동 가능한 접근성', '노포 감성의 정겨운 분위기로 모임이나 회식 장소로 적합']
  약점: ['저녁 시간대 웨이팅이 잦고 예약이 불가능함', '매장이 넓은 만큼 소음이 심하고 다소 시끄러운 편', '매장 앞 주차 공간이 매우 협소하여 공영주차장 이용 권장', '바쁜 시간대에는 직원들의 응대가 다소 불친절하거나 눈치가 보일 수 있음']
  요약: 신림역 인근의 노포 감성 고기집으로, 특히 우삼겹과 특제 소스의 조합이 매우 높은 평가를 받는 곳입니다. 다만 저녁 시간대에는 웨이팅이 길고 매장이 시끄러울 수 있어, 여유로운 식사를 원한다면 점심시간이나 이른 저녁 방문을 추천합니다.

■ 강강술래 신림점  (후기 5건 | 갈비)
  강점: ['프라이빗한 룸과 넓은 공간을 갖추어 가족 모임이나 회식에 적합함', '직원이 고기를 직접 구워주는 서비스로 편안한 식사가 가능함', '갈비탕 등 식사 메뉴의 가성비가 좋고 밑반찬이 정갈하게 제공됨', '주차 시설이 잘 갖춰져 있어 자차 이용이 편리함']
  약점: ['인기 있는 맛집이라 식사 시간대에는 웨이팅이 발생할 수 있음', '발렛 파킹 이용 시 별도의 관리비(2,000원)가 발생함', '일부 후기에서 특정 밑반찬(석박지)의 맛이 평범하다는 의견이 있음']
  요약: 강강술래 신림점은 쾌적한 룸 환경과 고기를 직접 구워주는 서비스 덕분에 가족 모임이나 어르신을 모시는 식사 자리에 매우 적합한 곳입니다. 고기뿐만 아니라 갈비탕 등 식사 메뉴의 만족도도 높으며, 주차 편의성까지 갖추고 있어 신림역 근처에서 추천할 만한 고기집입니다.

■ 왕주먹  (후기 5건 | 육류,고기)
  강점: ['직원이 연탄불 위에서 고기를 직접 구워주어 편리함', '첫 주문 3인분 이상 시 된장찌개와 돼지껍데기가 서비스로 제공됨'

In [27]:
run = "1787117045642"
summ = load(TOOL_DIR / f"{run}_info_summary.json")
for n, s in summ.items():
    print(f"{n:22} 강점{len(s['strengths'])} 약점{len(s['weaknesses'])}")

어사출또 신림역점              강점4 약점3
신림목포수산                 강점4 약점3
비타민수산                  강점3 약점3
육회바른연어 신림역점            강점4 약점2
비타민수산세꼬시               강점4 약점2
캐리비안회적                 강점4 약점3
길회센타                   강점4 약점2
막해                     강점4 약점3
해인어시장 신림본점             강점4 약점3
와사비                    강점3 약점3
올바른숙성회 신림2호점           강점2 약점1
신림광어                   강점4 약점2
광어&연어                  강점4 약점2
군산회집                   강점4 약점1
바다향                    강점4 약점2
통통회 신림점                강점4 약점3
어시장활어회 본점              강점4 약점3
탐나종합어시장 신림점            강점4 약점2


In [28]:
from collections import Counter

run = "1787116901784"   # 마산 합성동 피자
pool = load(TOOL_DIR / f"{run}_collect_places.json")

for i, p in enumerate(pool, 1):
    print(f"{i:2}위 {p['name']:26} | {p['category']}")

 1위 맥주를구한피자 마산합성점              | 음식점 > 양식 > 피자
 2위 피자덕 창원마산점                  | 음식점 > 양식 > 피자 > 피자덕
 3위 맘스터치 피자앤치킨 창원합성점           | 음식점 > 양식 > 피자 > 맘스터치 피자앤치킨
 4위 맘스피자 합성점                   | 음식점 > 양식 > 피자 > 맘스피자
 5위 피제리아부부                     | 음식점 > 양식 > 피자
 6위 미스터피자 석전점                  | 음식점 > 양식 > 피자 > 미스터피자
 7위 피나치공 마산석전점                 | 음식점 > 양식 > 피자 > 피나치공
 8위 반올림피자 경남마산양덕점              | 음식점 > 양식 > 피자 > 반올림피자
 9위 7번가피자 마산양덕점                | 음식점 > 양식 > 피자 > 7번가피자
10위 파파존스 마산회원점                 | 음식점 > 양식 > 피자 > 파파존스
11위 가득피자 합성점                   | 음식점 > 양식 > 피자
12위 피자에땅 합성점                   | 음식점 > 양식 > 피자 > 피자에땅
13위 빽보이피자 마산양덕점                | 음식점 > 양식 > 피자 > 빽보이피자
14위 오르새피자 회원점                  | 음식점 > 양식 > 피자
15위 피나치공 마산양덕점                 | 음식점 > 양식 > 피자 > 피나치공
16위 E맛있는피자 구암점                 | 음식점 > 양식 > 피자
17위 반올림피자 창원NC파크점              | 음식점 > 양식 > 피자 > 반올림피자
18위 아메리칸피자 마산양덕점               | 음식점 > 양식 > 피자 > 아메리칸피자
19위 반올림피자 1층 3루                | 음식점 > 양식 > 피자 > 반올

In [47]:
# 웹문서
items, status = await _naver_search("창원 횟집", kind="webkr", display=10)
for i in items:
    print(f"{i['title'][:60]}")
    print(f"   {i['description'][:100]}")
    print(f"   {i['link']}")
    print()

충무활어횟집 - 창원특례시청
   문의처 : 055-288-3133, 편의시설 : 없음 ; 경상남도 창원시 의류수거함 위치 총정리 ; 경상남도 창원시의 의류수거함 위치를 총정리했습니다. 창원시 정식명칭은 창원특례시
   https://www.changwon.go.kr/tour/index.do?foodProgramNo=193&menuCode=001_004004003000

어시장횟집 - 창원특례시청
   문의처 : 055-297-8311, 편의시설 : 없음 ; 창원시 전기차 충전기 위치정보 ; 위치 의창구 전기차 충전기 위치 진해구 전기차 충전기 위치 창원시 지역 목욕탕(찜질방) 
   https://www.changwon.go.kr/tour/index.do?foodProgramNo=147&menuCode=001_004004003000

대교횟집 - 창원특례시청
   문의처 : 055-265-0670, 편의시설 : 없음 ; 창원시 전기차 충전기 위치정보 목차 마산합포구 전기차 충전기 위치 마산회원구 전기차 충전기 위치 성산구 전기차 충전기 위치
   https://www.changwon.go.kr/tour/index.do?foodProgramNo=12&menuCode=001_004004002000

마산어시장횟집거리 - 창원관광
   소개 ; 마산어시장 횟집거리는 경상남도 창원시 마산합포구 동성동과 남성동의 어시장 일대에 바다 생선회를 전문으로 하는 식당들이 밀집한 지역으로 마산의 활어회는 바다와 인접한 지리적
   https://www.changwon.go.kr/tour/index.do?foodProgramNo=1&menuCode=001_004004001000

동방횟집 - 창원특례시청
   문의처 : 055-545-0409, 편의시설 : 없음 ; 창원시 전기차 충전기 위치정보 목차 마산합포구 전기차 충전기 위치 마산회원구 전기차 충전기 위치 성산구 전기차 충전기 위치
   https://www.changwon.go.kr/tour/index.do?foodPro

In [50]:
async def naver_web_diningcode(query: str, display: int = 30):
    items, _ = await _naver_search(query, kind="webkr", display=display)
    return [i for i in items if "diningcode.com" in i["link"]]

for it in await naver_web_diningcode("마산 합성동 피자"):
    print(it["title"])
    print(f"   {it['description'][:120]}")
    print(f"   {it['link']}\n")

맥주를구한피자 합성점 - 마산합성동 피자, 시카고피자 맛집
   맛5.0, 가격3.0, 친절3.0 바로 나온 피자는 역시 맛없없 맛있게 잘먹었습니다~ 화장실도 괜춚 08.12 (수) 영업시간: 17:30 - 23:50 시카고ㅡ페페로니(22,000원), 시카고-더블불고기(24,00
   https://www.diningcode.com/profile.php?rid=h77wtABGfHE2

마산 합성동 화덕피자 맛집 Top1
   마산 합성동 화덕피자 맛집 뜨라또리아 다젠나 (파스타, ★4.3) 등 1곳의 전체 순위,식당정보,방문자리뷰,사진 등을 확인하세요.
   https://www.diningcode.com/list.dc?query=%EB%A7%88%EC%82%B0%ED%95%A9%EC%84%B1%EB%8F%99%20%ED%99%94%EB%8D%95%ED%94%BC%EC%9E%90

마산 합성동 디트로이트피자 맛집 Top1
   마산 합성동 디트로이트피자 맛집 푸라닭 마산합성점(치킨) 등 1곳의 전체 순위,식당정보,방문자리뷰,사진 등을 확인하세요.
   https://www.diningcode.com/list.dc?query=%EB%A7%88%EC%82%B0%ED%95%A9%EC%84%B1%EB%8F%99%20%EB%94%94%ED%8A%B8%EB%A1%9C%EC%9D%B4%ED%8A%B8%ED%94%BC%EC%9E%90

마산 합성동 피자 맛집 Top13
   마산 합성동 피자 맛집 롤링파스타 마산합성점(파스타, ★4.6), 덕수파스타 마산합성점(스테이크, ★4.5), 뜨라또리아 다젠나 (파스타, ★4.3) 등 13곳의 전체 순위,식당정보,방문자리뷰,사진 등을 확인하세요.
   https://www.diningcode.com/list.dc?query=%EB%A7%88%EC%82%B0%ED%95%A9%EC%84%B1%EB%8F%99%20%ED%94%BC%EC%9E%90

PIZZERIA BUBU - 마산 화덕피자, 나폴리피자 맛집
   경상남도 창원시 마산회원구

In [52]:
import re

def parse_dc(desc: str) -> list[dict]:
    """다이닝코드 스니펫에서 '매장명 (카테고리, ★평점)' 패턴 추출"""
    return [
        {"name": m.group(1).strip(), "category": m.group(2).strip(),
         "rating": float(m.group(3))}
        for m in re.finditer(r"([가-힣A-Za-z0-9\s]+?)\s*\(([^,)]+),\s*★([\d.]+)\)", desc)
    ]

for it in await naver_web_diningcode("마산 합성동 피자"):
    print(parse_dc(it["description"]))

[]
[{'name': '마산 합성동 화덕피자 맛집 뜨라또리아 다젠나', 'category': '파스타', 'rating': 4.3}]
[]
[{'name': '마산 합성동 피자 맛집 롤링파스타 마산합성점', 'category': '파스타', 'rating': 4.6}, {'name': '덕수파스타 마산합성점', 'category': '스테이크', 'rating': 4.5}, {'name': '뜨라또리아 다젠나', 'category': '파스타', 'rating': 4.3}]
[]
[{'name': '마산 합성동 술집 맛집 가치관', 'category': '술집', 'rating': 5.0}, {'name': '챠밍복덕', 'category': '술집', 'rating': 3.5}, {'name': '난데모', 'category': '스키야키', 'rating': 3.6}]
[{'name': '마산 합성동 파스타 맛집 롤링파스타 마산합성점', 'category': '파스타', 'rating': 4.6}, {'name': '덕수파스타 마산합성점', 'category': '스테이크', 'rating': 4.5}, {'name': '베비도 마산합성점', 'category': '멕시코음식', 'rating': 4.5}]
[]
[{'name': '마산 합성동 시카고피자 맛집 맥주를구한피자 합성점', 'category': '피자', 'rating': 4.0}]
[]
[]
[]
[]
[{'name': '마산 화덕피자 맛집 피제리아부부', 'category': '화덕피자', 'rating': 3.7}, {'name': '오오브', 'category': '파스타', 'rating': 4.3}, {'name': '비바스파게티', 'category': '파스타', 'rating': 4.2}]
[]
[]
[]
[{'name': '마산 합성동 맥주 맛집 동키치킨 마산합성점', 'category': '치킨', 'rating': 4.5}, {'name': '노군꼬치 합성점', 'cate

In [54]:
targets = {
    "뜨라또리아 다젠나": ["뜨라또리아 다젠나", "마산 뜨라또리아", "합성동 파스타", "마산 화덕피자"],
    "개성순대":        ["개성순대", "마산 개성순대", "가포 순대", "마산합포구 순대"],
    "우동한그릇":       ["우동한그릇", "마산 우동한그릇", "가포 우동"],
    "귀산옥 해물칼국수":  ["귀산옥", "창원 귀산옥", "귀산 칼국수", "귀산동 해물칼국수"],
    "거영수산in해녀의집": ["거영수산", "해녀의집 창원", "창원 거영수산", "진동 해녀의집"],
}

for answer, queries in targets.items():
    print(f"\n{'='*70}\n■ {answer}")
    for q in queries:
        places, _ = _kakao_keyword(q)
        hit = [p for p in places if any(t in p["name"] or p["name"] in t
                                        for t in [answer, answer.split()[0]])]
        if hit:
            p = hit[0]
            rank = places.index(p) + 1
            print(f"  ✓ '{q}' → {rank}위 | {p['name']} | {p['category']}")
            print(f"      {p['address']}")
        else:
            print(f"  ✗ '{q}' → {[x['name'] for x in places[:3]]}")


■ 뜨라또리아 다젠나
  ✓ '뜨라또리아 다젠나' → 1위 | 뜨라또리아 다젠나 | 음식점 > 양식 > 이탈리안
      경남 창원시 마산회원구 3.15대로 728
  ✓ '마산 뜨라또리아' → 1위 | 뜨라또리아 다젠나 | 음식점 > 양식 > 이탈리안
      경남 창원시 마산회원구 3.15대로 728
  ✓ '합성동 파스타' → 1위 | 뜨라또리아 다젠나 | 음식점 > 양식 > 이탈리안
      경남 창원시 마산회원구 3.15대로 728
  ✗ '마산 화덕피자' → ['비바 경남대점', '포넬로450', '오오브']

■ 개성순대
  ✓ '개성순대' → 1위 | 개성순대 | 음식점 > 한식 > 순대
      경남 창원시 마산합포구 가포로 686
  ✓ '마산 개성순대' → 1위 | 개성순대 | 음식점 > 한식 > 순대
      경남 창원시 마산합포구 가포로 686
  ✗ '가포 순대' → ['전남친순대 월영경남대점', '정성순대 귀산점', '순대와 녹두빈대떡']
  ✓ '마산합포구 순대' → 1위 | 개성순대 | 음식점 > 한식 > 순대
      경남 창원시 마산합포구 가포로 686

■ 우동한그릇
  ✓ '우동한그릇' → 1위 | 우동한그릇 | 음식점 > 일식 > 돈까스,우동
      경남 창원시 마산합포구 가포로 706
  ✓ '마산 우동한그릇' → 1위 | 우동한그릇 | 음식점 > 일식 > 돈까스,우동
      경남 창원시 마산합포구 가포로 706
  ✗ '가포 우동' → ['하루우동 마산가포점', '한신우동 경남대점', '강성현돈가스']

■ 귀산옥 해물칼국수
  ✓ '귀산옥' → 1위 | 귀산옥 해물칼국수 | 음식점 > 한식 > 국수 > 칼국수
      경남 창원시 성산구 삼귀로 131
  ✓ '창원 귀산옥' → 1위 | 귀산옥 해물칼국수 | 음식점 > 한식 > 국수 > 칼국수
      경남 창원시 성산구 삼귀로 131
  ✓ '귀산 칼국수' → 2위 | 귀산옥 해물칼국수 | 음식점 > 한식 > 국수 > 칼국수
      경남 

In [57]:
places, _ = _kakao_keyword("가포 맛집")
from collections import Counter
addrs = [p["address"].split() for p in places]
print(Counter(a[2] for a in addrs if len(a) > 2))

for q in ["마산합포구 맛집", "마산합포구 순대", "마산합포구 우동", "마산합포구 칼국수"]:
    ps, _ = _kakao_keyword(q)
    print(f"\n{q}")
    for p in ps[:6]:
        print(f"   {p['name']:24} | {p['address']}")

Counter({'마산합포구': 15})

마산합포구 맛집
   우동한그릇                    | 경남 창원시 마산합포구 가포로 706
   브라운핸즈 마산점                | 경남 창원시 마산합포구 가포순환로 109
   신포장어                     | 경남 창원시 마산합포구 수산2길 113
   지중해                      | 경남 창원시 마산합포구 가포로 501-201
   원조반동굴구이                  | 경남 창원시 마산합포구 구산면 해양관광로 1512
   이층횟집                     | 경남 창원시 마산합포구 진동면 미더덕로 345-1

마산합포구 순대
   개성순대                     | 경남 창원시 마산합포구 가포로 686
   통큰순대 본점                  | 경남 창원시 마산합포구 진북면 삼진의거대로 514
   정성순대 마산점                 | 경남 창원시 마산합포구 천하장사로 11
   전남친순대 월영경남대점             | 경남 창원시 마산합포구 월영동1길 6
   찐한순대국밥 마산오동점             | 경남 창원시 마산합포구 오동서1길 2
   오늘수육순대국 마산중앙점            | 경남 창원시 마산합포구 장군천로 42

마산합포구 우동
   우동한그릇                    | 경남 창원시 마산합포구 가포로 706
   52돈까스                    | 경남 창원시 마산합포구 월영북15길 21
   카츠혼                      | 경남 창원시 마산합포구 월영동1길 8
   하루우동 마산가포점               | 경남 창원시 마산합포구 가포로 267
   동양카츠 마산점                 | 경남 창원시 마산합포구 용마로 84
   피노키오                     | 경남 창원시 마산합포구 동서북14길 

In [59]:
places = []
seen = set()
for page in range(1, 4):          # 15 × 3 = 45가 상한
    batch, is_end = _kakao_keyword("가포 맛집", page=page)
    for p in batch:
        if p["id"] not in seen:
            seen.add(p["id"])
            places.append(p)
    print(f"page {page}: +{len(batch)}건 (누적 {len(places)}) is_end={is_end}")
    if is_end:
        break

print(f"\n총 {len(places)}개\n")

targets = ["우동한그릇", "개성순대", "등대집", "동굴집", "매료테이블"]
for i, p in enumerate(places, 1):
    star = " ★" if any(t in p["name"] or p["name"] in t for t in targets) else ""
    cat = p["category"].split(">")[-1].strip()
    print(f"{i:2}위 {p['name']:26} | {cat:12} | {p['address']}{star}")

page 1: +15건 (누적 15) is_end=False
page 2: +15건 (누적 30) is_end=False
page 3: +15건 (누적 45) is_end=True

총 45개

 1위 브라운핸즈 마산점                  | 커피전문점        | 경남 창원시 마산합포구 가포순환로 109
 2위 지중해                        | 카페           | 경남 창원시 마산합포구 가포로 501-201
 3위 옛날영도집                      | 장어           | 경남 창원시 마산합포구 가포해안길 37
 4위 동굴집                        | 오리           | 경남 창원시 마산합포구 가포해안길 35 ★
 5위 명품갈비토반                     | 갈비           | 경남 창원시 마산합포구 가포신항북로 20
 6위 우주슈퍼                       | 카페           | 경남 창원시 마산합포구 가포동 38
 7위 해마루                        | 카페           | 경남 창원시 마산합포구 가포로 506
 8위 박배철베이커리 해유                 | 제과,베이커리      | 경남 창원시 마산합포구 가포본동5길 4
 9위 해월수                        | 한식           | 경남 창원시 마산합포구 가포로 501-200
10위 베이커리카페빵내음                  | 제과,베이커리      | 경남 창원시 마산합포구 가포로 374-172
11위 안민옥 마산직영점                  | 샤브샤브         | 경남 창원시 마산합포구 가포신항북2길 5-2
12위 가포장어구이                     | 장어           | 경남 창원시 마산합포구 가포해안길 81
13위 더지노                        | 이탈리안 

In [60]:
from collections import Counter
print(Counter(p["address"].split()[2] for p in places if len(p["address"].split()) > 2))
print(Counter(p["category"].split(">")[-1].strip() for p in places).most_common(10))

Counter({'마산합포구': 45})
[('카페', 8), ('장어', 6), ('제과,베이커리', 4), ('한식', 4), ('육류,고기', 4), ('양식', 3), ('갈비', 2), ('이탈리안', 2), ('커피전문점', 1), ('오리', 1)]


In [63]:
items, status = await _naver_search("다이닝코드 창원 가포 맛집", kind="webkr", display=20)
for i in items:
    if "diningcode" in i["link"]:
        print(i["title"])
        print(f"  {i['description']}")
        print(f"  {i['link']}\n")

창원 장어구이 맛집 Top100
  창원 장어구이 맛집 가포옛날영도집 (장어구이, ★4.4), 본길 (장어덮밥, ★4.2), 대도장어 (장어, ★4.8) 등 300곳의 전체 순위,식당정보,방문자리뷰,사진 등을 확인하세요.
  https://www.diningcode.com/list.dc?query=%EC%B0%BD%EC%9B%90%20%EC%9E%A5%EC%96%B4%EA%B5%AC%EC%9D%B4

가포경양식 - 마산 경양식, 돈까스 맛집
  맛5.0, 가격5.0, 친절5.0 가게 전용 주차장은 없었지만 가게 앞과 근처 대로에 주차 가능해서 차량 방문하기 좋았어요. 수제돈까츠 고기도 질기지 않고 적당히 식감이 있었고, 소스는 살짝 달달해서 아이...
  https://www.diningcode.com/profile.php?rid=Koy2YS7sou6G

가포장어구이 - 마산 장어구이 맛집
  07.25 (토) 영업시간: 10:00 - 21:30 장어양념구이 (大)(60,000원), 장어소금구이 (大)(60,000원) 환경친화적, 정갈한
  https://www.diningcode.com/profile.php?rid=y9Q9umKKsuPv

경상남도 창원시 마산합포구 가포본동 맛집 Top18
  경상남도 창원시 마산합포구 가포본동 맛집 박배철베이커리 해유 (베이커리카페, ★4.1), 디어피노 (파스타, ★4), 가포경양식 (경양식, ★4) 등 18곳의 전체 순위,식당정보,방문자리뷰,사진 등을 확인하세요.
  https://www.diningcode.com/list.dc?query=%EA%B2%BD%EC%83%81%EB%82%A8%EB%8F%84%20%EC%B0%BD%EC%9B%90%EC%8B%9C%20%EB%A7%88%EC%82%B0%ED%95%A9%ED%8F%AC%EA%B5%AC%20%EA%B0%80%ED%8F%AC%EB%B3%B8%EB%8F%992%EA%B8%B8

가포영도장어구이전문점 - 마산 장어구이 맛집
  맛5.0, 가격5.0, 친절4.3 양념장어구

In [65]:
queries = [
    "가포동 맛집 diningcode",
    "마산 가포 맛집 순위",
    "가포 맛집 Top",
    "diningcode 마산합포구 가포",
]
for q in queries:
    items, _ = await _naver_search(q, kind="webkr", display=20)
    dc = [i for i in items if "diningcode" in i["link"]]
    print(f"\n=== {q} ({len(dc)}건) ===")
    for i in dc:
        print(f"  {i['title'][:60]}")
        print(f"    {i['description'][:200]}")


=== 가포동 맛집 diningcode (20건) ===
  경남 창원시 마산합포구 가포동 맛집 Top45
    경남 창원시 마산합포구 가포동 맛집 브라운핸즈 마산점(카페, ★3.5), 동굴집 (오리불고기, ★4.9), 메르칼므 (Mer Calme) (카페, ★5) 등 45곳의 전체 순위,식당정보,방문자리뷰,사진 등을 확인하세요.
  가포장어구이 - 마산 장어구이 맛집
    07.25 (토) 영업시간: 10:00 - 21:30 장어양념구이 (大)(60,000원), 장어소금구이 (大)(60,000원) 환경친화적, 정갈한
  가포수산 - 강서구 대게, 킹크랩 맛집
    08.02 (일) 영업시간: 06:30 - 20:00 A급 러시아 블루 킹크랩(마리) (1.8kg~2.5kg)(171000~237500), A급 러시아 레드 킹크랩(마리) (6.7kg~7kg)(737000~770000), 흰다리새우 (대 사이즈 10미)(10,000
  가포경양식 - 마산 경양식, 돈까스 맛집
    맛5.0, 가격5.0, 친절5.0 가게 전용 주차장은 없었지만 가게 앞과 근처 대로에 주차 가능해서 차량 방문하기 좋았어요. 수제돈까츠 고기도 질기지 않고 적당히 식감이 있었고, 소스는 살짝 달달해서 아이...
  가포영도장어구이전문점 - 마산 장어구이 맛집
    맛5.0, 가격5.0, 친절4.3 양념장어구이 맛있어요. 장어탕수도 새콤달콤 하니 맛있습니다. 07.23 (목) 영업시간: 11:00 - 21:00 양념,소금장어구이 (소), 유린장어.탕수장어.깐풍장어(35,000원), 식사 (장어국,밥)(3,000원), 양념,소금장
  토담갈비찜 가포점 - 마산 한식, 갈비찜 맛집
    맛3.0, 가격3.0, 친절3.0 주차장은 따로 없지만 위치가 가포안쪽 해안변공원 근처라 갓길에 주차하기 좋아요. 보통맛은 매콤달달한 소스로 밥이랑 같이먹으면 맛있습니다. 기본찬으로 나오는 간장?에 절여진...
  안민옥 가포직영점 - 마산 칼국수, 샤브칼국수 맛집
    맛5.0, 가격5.0, 친

In [68]:
@_track("kakao")
def _kakao_keyword(
    query: str,
    page: int = 1,
    category_code: str = "",
    x: str | float | None = None,
    y: str | float | None = None,
    radius: int | None = None,
) -> tuple[list[dict], bool]:
    """카카오 로컬 키워드 검색 1페이지. (결과, is_end) 반환.

    카카오는 상호명·카테고리·태그 문자열 매칭이라 서술형 검색어에 약하다.
    페이지당 15개, 전체 노출 가능 문서는 45개가 상한.

    x, y를 주면 그 좌표 기준으로 검색하고 응답에 distance(m)가 붙는다.
    행정동 경계와 무관하게 물리적 거리로 잡히므로, 생활권 통칭("가포")과
    행정동("덕동동")이 어긋나는 지역에서 커버리지가 크게 늘어난다.
    radius는 미터 단위, 최대 20000.
    """
    params = {"query": query, "size": 15, "page": page}
    if category_code:
        # 카카오는 단일 코드만 받는다. 여러 개가 오면 첫 번째만 쓴다.
        params["category_group_code"] = category_code.split(",")[0].strip()
    if x is not None and y is not None:
        params["x"] = str(x)
        params["y"] = str(y)
        if radius:
            params["radius"] = min(int(radius), 20000)

    resp = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers={"Authorization": f"KakaoAK {os.environ['KAKAO_REST_API_KEY']}"},
        params=params,
        timeout=5,
    )
    resp.raise_for_status()
    body = resp.json()

    places = [
        {
            "id": d["id"],
            "name": d["place_name"],
            "category": d["category_name"],
            "address": d["road_address_name"] or d["address_name"],
            "phone": d.get("phone", ""),
            "url": d["place_url"],
            "x": float(d["x"]),                              # 경도
            "y": float(d["y"]),                              # 위도
            "distance": int(d["distance"]) if d.get("distance") else None,
            "source": "kakao_local",
        }
        for d in body["documents"]
    ]
    return places, body["meta"]["is_end"]

In [69]:
places, _ = _kakao_keyword("가포 맛집")
cx = sum(float(p["x"]) for p in places) / len(places)
cy = sum(float(p["y"]) for p in places) / len(places)
# 이 좌표 + radius=2000으로 "순대", "우동", "칼국수" 등을 재검색

In [71]:
# 1차: 원 검색어로 중심점 확보
base, _ = _kakao_keyword("가포 맛집")
cx = sum(p["x"] for p in base) / len(base)
cy = sum(p["y"] for p in base) / len(base)
print(f"중심: {cx:.6f}, {cy:.6f}  ({len(base)}개 기준)\n")

# 2차: 그 좌표 반경 2km에서 업종별 재검색
targets = ["우동한그릇", "개성순대", "매료테이블", "등대집", "동굴집"]
for q in ["맛집", "순대", "우동", "식당", "국수"]:
    ps, _ = _kakao_keyword(q, x=cx, y=cy, radius=2000)
    hit = [p for p in ps if any(t in p["name"] or p["name"] in t for t in targets)]
    print(f"[{q}] {len(ps)}개" + (f"  ★ {[p['name'] for p in hit]}" if hit else ""))
    for p in ps[:5]:
        print(f"    {p['name']:24} {p['distance']:>5}m | {p['address']}")
    print()

중심: 128.579517, 35.161624  (15개 기준)

[맛집] 15개
    홀리벨리                       500m | 경남 창원시 마산합포구 가포본동2길 18
    박배철베이커리 해유                 684m | 경남 창원시 마산합포구 가포본동5길 4
    김앤주과자점                     718m | 경남 창원시 마산합포구 가포본동7길 9
    바보형제쭈꾸미 가포점                635m | 경남 창원시 마산합포구 가포로 290
    가포경양식                      571m | 경남 창원시 마산합포구 가포본동6길 4-13

[순대] 1개
    정성순대 귀산점                  1845m | 경남 창원시 성산구 삼귀로 131

[우동] 6개
    하루우동 마산가포점                 975m | 경남 창원시 마산합포구 가포로 267
    귀산돈까스 귀산본점                1925m | 경남 창원시 성산구 삼귀로 177
    낙원우동                       624m | 경남 창원시 마산합포구 가포로 429
    어울림하우스                    1126m | 경남 창원시 마산합포구 가포로 490-1
    가포술집                       638m | 경남 창원시 마산합포구 가포본동1길 18

[식당] 15개  ★ ['동굴집']
    바보형제쭈꾸미 가포점                635m | 경남 창원시 마산합포구 가포로 290
    가포경양식                      571m | 경남 창원시 마산합포구 가포본동6길 4-13
    명품갈비토반                     871m | 경남 창원시 마산합포구 가포신항북로 20
    디어피노                       677m | 경남 창원시 마산합포구 가포본동7길 5
    콜리나        